# Declared and assessed VAT (ICMS) debt in São Paulo's tax arrears

**Research front:** `projects/macro` · **Article slug:** `icms_declarado_autuado` · **Task:** PESQ-AVD-01 (article notebook).

**Question.** In São Paulo's enrolled tax debt (*dívida ativa*) for the state VAT (ICMS), do the debt that the taxpayer declared and did not pay (*declared*) and the debt raised by the tax administration in an infringement notice (*assessed*) differ in size, concentration, payment and recovery enough to require separate treatment in studies of VAT compliance and debt collection?

**What this notebook does.** All of the article's analysis lives here. It reads the CEMEPI database (DuckDB, read-only), recomputes every number, and writes the figures (Plotly + Kaleido, with the plotted data as CSV alongside), the tables and `numeros.tex` to `projects/macro/manuscript/icms_declarado_autuado/`. The article's LaTeX (next task) only includes these outputs.

**Rules.** Only aggregates leave the notebook: no name, CPF, CNPJ, CNPJ root or individual CDA (debt certificate) appears in any output, table or figure. No 2026 data: portfolio snapshots stop in December 2025 and payments on 31 Dec 2025. Interpretations belong to the author and are flagged **[ÉTORE DECIDE]**.

## 0. Data version and environment

**Layers (docs/ESTRUTURA.md §2).** The article does not read the lab (L4). The shared L3 package (`pipelines/divida_ativa/src/divida_ativa`) does not exist on `main` yet, and there is no L2 manifest for an ICMS panel in `data_manifests/` (only `cnpj_enrich`). Data are therefore read directly from `cemepi.duckdb`, attached **read-only** (`ATTACH ... (READ_ONLY)`) to an in-memory DuckDB connection; temporary tables live in that connection and spill goes to a system temporary folder that is removed at the end.

**Pinned version.** `extracao=2026-03` (PGE-SP *Inteligência Fiscal* dump). The cell below checks the database file size and `mtime` against pinned values; if the database changes, the notebook stops instead of silently producing new numbers. The path comes from an environment variable (`CEMEPI_DUCKDB`, or `CEMEPI_DATA_ROOT` + `api_ctf/cemepi.duckdb`), never from a hard-coded path.

In [ ]:
import os, json, time, shutil, tempfile, platform
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import duckdb
import scipy
from scipy import stats
import sklearn
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score
import plotly
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

def find_repo(start: Path = Path.cwd()) -> Path:
    for p in [start, *start.parents]:
        if (p / "AGENTS.md").exists() and (p / "pyproject.toml").exists():
            return p
    raise RuntimeError("repository root not found")

REPO = find_repo()
SLUG = "icms_declarado_autuado"
OUT = REPO / "projects/macro/manuscript" / SLUG
FIG, TAB = OUT / "figures", OUT / "tables"

# ---- pinned data version ----
EXTRACAO = "2026-03"
DB = Path(os.environ["CEMEPI_DUCKDB"]) if os.environ.get("CEMEPI_DUCKDB") else \
     Path(os.environ["CEMEPI_DATA_ROOT"]) / "api_ctf" / "cemepi.duckdb"
DB_EXPECTED = {"bytes": 39_812_870_144, "mtime_epoch": 1_790_177_521}   # checked on 2026-10-27
st = DB.stat()
DB_SEEN = {"bytes": st.st_size, "mtime_epoch": int(st.st_mtime)}
assert DB_SEEN == DB_EXPECTED, f"database changed: {DB_SEEN} != {DB_EXPECTED}"

# ---- scope ----
SNAP = (2025, 12)                    # cross-section: December 2025 snapshot (no 2026)
SNAP_DATE = "2025-12-01"             # reference date for age in the snapshot
YEAR_MIN, YEAR_MAX = 2016, 2025      # CDA x year panel
PAY_END = "2026-01-01"               # GARE payments dated < 2026-01-01
ENROL_MIN, ENROL_MAX = "1960-01-01", "2026-03-31"   # enrolment dates outside this range are set to null (data-entry errors)
PROGRAMMES = [("PEP 2019", "2019-07-01", "2019-12-31"), ("Acordo Paulista 2024", "2024-04-01", "2024-12-31")]
SEED = 2026
np.random.seed(SEED)

ICMS_TYPES = ["ICMS Declarado", "ICMS Autuação", "ICMS Autuação Simples Nacional", "ICMS Fecoep", "ICMS Alteração de GIA"]
TYPE_EN = {"ICMS Declarado": "Declared", "ICMS Autuação": "Assessed", "ICMS Autuação Simples Nacional": "Assessed, Simples Nacional",
           "ICMS Fecoep": "Fecoep", "ICMS Alteração de GIA": "GIA amendment"}
TYPES_SQL = ", ".join(f"'{t}'" for t in ICMS_TYPES)
ORIGIN_SQL = '''CASE WHEN TIPO_DEBITO = 'ICMS Declarado' THEN 'declared'
                     WHEN TIPO_DEBITO LIKE 'ICMS Autua%' THEN 'assessed' ELSE 'other' END'''
# Debtor CNPJ normalised as in pipelines/cnpj_enrich (upper case, punctuation removed, 12-13 digits left-padded to 14).
# Legal entity (PJ): normalised CNPJ with 14 characters. CPF_DEVEDOR and NOME_DEVEDOR are never read.
_CL = "REGEXP_REPLACE(UPPER(CAST(CNPJ_DEVEDOR AS VARCHAR)), '[^0-9A-Z]', '', 'g')"
CNPJ_SQL = f"CASE WHEN REGEXP_FULL_MATCH({_CL}, '[0-9]{{12,13}}') THEN LPAD({_CL}, 14, '0') ELSE NULLIF({_CL}, '') END"
PJ_SQL = f"LENGTH({CNPJ_SQL}) = 14"
DATE_SQL = (f"CASE WHEN TRY_CAST(LEFT(DATA_INSCRICAO, 10) AS DATE) BETWEEN DATE '{ENROL_MIN}' AND DATE '{ENROL_MAX}' "
            f"THEN TRY_CAST(LEFT(DATA_INSCRICAO, 10) AS DATE) END")

TMP = Path(tempfile.mkdtemp(prefix="avd_duckdb_"))
con = duckdb.connect(config={"memory_limit": "2500MB", "threads": 4, "temp_directory": str(TMP),
                             "preserve_insertion_order": False})
con.execute(f"ATTACH '{DB}' AS src (READ_ONLY)")

# the manuscript folder starts empty on every run (outputs of this notebook only)
for d in (FIG, TAB):
    if d.exists():
        shutil.rmtree(d)
    d.mkdir(parents=True)

ENV = {"python": platform.python_version(), "duckdb": duckdb.__version__, "pandas": pd.__version__,
       "numpy": np.__version__, "scipy": scipy.__version__, "sklearn": sklearn.__version__,
       "plotly": plotly.__version__, "extracao": EXTRACAO, "db": DB_SEEN, "seed": SEED,
       "run_at": datetime.now(timezone.utc).astimezone().isoformat(timespec="seconds")}
print(json.dumps(ENV, indent=1))

### 0.1 Output helpers

Every number quoted in the article is registered in `N` under a key and becomes a `\newcommand` in `numeros.tex` at the end. Numbers are written in English format (comma as thousands separator, point as decimal mark). Every figure is written as PNG and SVG by Kaleido, with a CSV of the plotted data alongside. Every table is written as CSV and as LaTeX (`booktabs`).

In [ ]:
N: dict[str, tuple[str, str]] = {}      # key -> (formatted text, source)
RAW: dict[str, float] = {}              # key -> unrounded value (used only for the draft comparison)

def fmt(x, nd=1):
    return f"{x:,.{nd}f}"

def reg(key: str, value, nd=1, source=""):
    assert key.isalpha(), key
    txt = fmt(value, nd) if isinstance(value, (int, float, np.integer, np.floating)) else str(value)
    N[key] = (txt, source)
    if isinstance(value, (int, float, np.integer, np.floating)):
        RAW[key] = float(value)
    return value

CEMEPI = {"deep_teal": "#154C56", "teal": "#186D7F", "mid_teal": "#267586", "sand": "#BD9869",
          "green": "#5EA65B", "cream": "#FFF4E5", "grey": "#8A9499"}
COL = {"assessed": CEMEPI["deep_teal"], "declared": CEMEPI["sand"], "other": CEMEPI["grey"]}
pio.templates["cemepi"] = go.layout.Template(layout=go.Layout(
    font=dict(family="Montserrat, Arial, sans-serif", size=11, color=CEMEPI["deep_teal"]),
    paper_bgcolor="white", plot_bgcolor="white", margin=dict(l=10, r=20, t=40, b=10, autoexpand=True),
    xaxis=dict(showgrid=False, zeroline=False, showline=True, linecolor=CEMEPI["deep_teal"], ticks="outside", automargin=True),
    yaxis=dict(showgrid=True, gridcolor="#E6E6E6", zeroline=False, showline=True, linecolor=CEMEPI["deep_teal"],
               ticks="outside", automargin=True),
    legend=dict(bgcolor="rgba(0,0,0,0)", orientation="h", y=-0.2)))
pio.templates.default = "cemepi"

def save_fig(fig, name: str, data: pd.DataFrame, width=700, height=330):
    fig.write_image(FIG / f"{name}.png", width=width, height=height, scale=4)
    fig.write_image(FIG / f"{name}.svg", width=width, height=height)
    data.to_csv(FIG / f"{name}.csv", index=False)
    return fig

def save_tab(df: pd.DataFrame, name: str, fmts: dict | None = None, caption: str = ""):
    df.to_csv(TAB / f"{name}.csv", index=False)
    out = df.copy()
    for c, nd in (fmts or {}).items():
        out[c] = out[c].map(lambda v: "" if pd.isna(v) else fmt(v, nd))
    esc = lambda x: str(x).replace("_", r"\_").replace("%", r"\%").replace("$", r"\$").replace("&", r"\&")
    cols = " & ".join(esc(c) for c in out.columns)
    body = "\n".join(" & ".join(esc(v) for v in r) + r" \\" for r in out.itertuples(index=False))
    tex = ("% generated by projects/macro/notebooks/articles/icms_declarado_autuado.ipynb; do not edit by hand\n"
           + (f"% {caption}\n" if caption else "")
           + r"\begin{tabular}{" + "l" + "r" * (len(out.columns) - 1) + "}\n\\toprule\n" + cols + r" \\" + "\n\\midrule\n"
           + body + "\n\\bottomrule\n\\end{tabular}\n")
    (TAB / f"{name}.tex").write_text(tex, encoding="utf-8")
    return df

def q(sql: str) -> pd.DataFrame:
    return con.execute(sql).df()

## 1. ICMS as a subnational VAT

ICMS (*Imposto sobre Circulação de Mercadorias e Serviços*) is levied by each Brazilian state on the sale of goods, electricity, interstate and intermunicipal transport and communications, with credit for the tax paid on inputs. An OECD Economics Department working paper describes it as "a State-level VAT on the sale of goods, electric power, interstate and intermunicipal transportation and communications" (Arnold et al., 2025, OECD Economics Department Working Paper No. 1848). The OECD classification of taxes used in *Revenue Statistics in Latin America and the Caribbean* places under heading 5111, value-added taxes, all general consumption taxes charged on value added, "irrespective of the method of deduction and the stages at which the taxes are levied" (OECD Interpretative Guide, §61). ICMS is recognised internationally as a value-added tax: beyond the OECD sources, peer-reviewed studies refer to it as "Brazil’s state value-added tax (ICMS)" (Gomes and Balaniuk, 2024, *Revista de Administração Pública*) and as "the state VAT (ICMS)" (Cabello, Gaio and Rezende, 2020, *Advances in Scientific and Applied Accounting*); both DOIs were checked on Crossref and doi.org on 27 Oct 2026. Throughout the notebook the tax is called **VAT (ICMS)**; raw database labels keep the Portuguese names (e.g. `ICMS Declarado`) because they are the values of `TIPO_DEBITO`.

**Why this matters for the question.** In the VAT, the amount due is in principle documented by the invoice trail (Pomeranz, 2015) and self-reported by the taxpayer; non-compliance therefore splits into *declared and unpaid* tax and tax that only an audit reveals (Keen and Smith, 2006). The compliance literature distinguishes third-party-reported from self-reported bases (Kleven et al., 2011), and the starting point is the classical evasion model (Allingham and Sandmo, 1972). In São Paulo's enrolled VAT debt, the two doors are visible in the data: `ICMS Declarado` (declared) and `ICMS Autuação` (assessed). This notebook describes them; it does not test any of these models.

## 2. Data and definitions

- **Sources.** `DEBITO`: monthly snapshots of the enrolled portfolio (one row per CDA and snapshot, fields `ANO_EXTRACAO`, `MES_EXTRACAO`). `ARRECADACAO`: GARE payments linked to the CDA by `ID_DEBITO`.
- **VAT (ICMS) debt types.** The five raw values of `TIPO_DEBITO` starting with ICMS. **Origin:** *declared* = `ICMS Declarado`; *assessed* = `ICMS Autuação` and `ICMS Autuação Simples Nacional`; *other* = `ICMS Fecoep` and `ICMS Alteração de GIA`, kept in totals and not interpreted. Where a table reports raw types, `ICMS Autuação` ("Assessed") excludes Simples Nacional. **[ÉTORE DECIDE]** confirm with PGE that *declared* corresponds to declared-and-unpaid debt (self-assessment, *homologação*) and not to another modality.
- **Legal entity (PJ).** Debtor CNPJ with 14 characters after normalisation (as in the CNPJ enrichment pipeline); the CPF and name columns of `DEBITO` are never read, so a row with both a CPF and a CNPJ would count as a legal entity. The **debtor** level is the CNPJ root (first 8 digits), used only inside aggregations.
- **Cross-section.** December 2025 snapshot. A CDA with more than one row in the snapshot keeps its largest value; null or non-positive values are dropped.
- **CDA × year panel (2016–2025, PJ).** For each CDA and year in which it appears, the **year's stock** is its value in the last snapshot of the year in which it is present; **paid in the year** is the sum of `VALOR_RECEITA` over GARE payments dated in that year. The CDA's type and enrolment date are those of the last snapshot in which it appears. Payments in years in which the CDA is in no snapshot are not in the panel.

## 3. Limitations of the `DEBITO` table (stated, not corrected)

**Method.** Each year of `DEBITO` is read once. For each monthly snapshot from 2016-01 to 2025-12, the notebook counts rows and distinct CDAs of VAT (ICMS) debt owed by legal entities and sums their value. A snapshot is flagged as:
- **missing**, if it has no rows;
- **partial?**, if its number of CDAs is below 80% of the centred 7-snapshot rolling median (missing snapshots do not enter the median): $n_t < 0.8\,\operatorname{med}(n_{t-3},\dots,n_{t+3})$;
- **duplicated rows**, if it has more rows than distinct CDAs.

**Known from the earlier diagnosis** (to be confirmed below): missing snapshots in 2018-12 and 2020-10 to 2020-12; a partial snapshot in 2024-12; duplicated rows in June and July 2025. **How this affects the article:** (i) the year's stock uses the last snapshot of the year in which the CDA appears, so in 2018 and 2020 it comes from November and September, and in 2024 the partial December snapshot may replace the December value by an earlier one for part of the CDAs; (ii) duplicated rows are counted once; (iii) nothing is imputed. The December 2025 cross-section should be unaffected by any of these problems, which is checked below. All results that depend on the yearly panel must be re-run once the `DEBITO` gaps are restored.

In [ ]:
# One pass per year over DEBITO (external drive): the year's VAT (ICMS) rows of legal entities go to a temporary table,
# from which come the per-snapshot counts (this section) and the year-end stock per CDA (panel, section 6).
t0 = time.time()
con.execute("CREATE OR REPLACE TEMP TABLE ye (year SMALLINT, ID_DEBITO BIGINT, stock DOUBLE, tipo VARCHAR, dt_insc DATE, last_month TINYINT, cnpj VARCHAR)")
qual_parts = []
for y in range(YEAR_MIN, YEAR_MAX + 1):
    con.execute(f'''
    CREATE OR REPLACE TEMP TABLE yr AS
    SELECT MES_EXTRACAO AS month, ID_DEBITO, VALOR_SEM_HONORARIOS AS value, TIPO_DEBITO AS tipo, {DATE_SQL} AS dt_insc,
           {CNPJ_SQL} AS cnpj
    FROM src.DEBITO
    WHERE ANO_EXTRACAO = {y} AND MES_EXTRACAO BETWEEN 1 AND 12 AND TIPO_DEBITO IN ({TYPES_SQL}) AND {PJ_SQL}''')
    qual_parts.append(q(f"SELECT {y} AS year, month, COUNT(*) AS rows, COUNT(DISTINCT ID_DEBITO) AS cdas, "
                        f"SUM(value) / 1e9 AS value_bn FROM yr GROUP BY month"))
    con.execute(f'''
    INSERT INTO ye
    SELECT {y}, ID_DEBITO, arg_max(value, month), arg_max(tipo, month), MIN(dt_insc), MAX(month), arg_max(cnpj, month)
    FROM yr GROUP BY ID_DEBITO''')
con.execute("DROP TABLE yr")
qual = pd.concat(qual_parts)
grid = pd.DataFrame([(y, m) for y in range(YEAR_MIN, YEAR_MAX + 1) for m in range(1, 13)], columns=["year", "month"])
qual = grid.merge(qual, on=["year", "month"], how="left").sort_values(["year", "month"]).reset_index(drop=True)
qual["snapshot"] = qual.year.astype(str) + "-" + qual.month.map("{:02d}".format)
qual["ref_median7"] = qual.cdas.rolling(7, center=True, min_periods=3).median()
qual["status"] = np.select(
    [qual.cdas.isna(), qual.cdas < 0.8 * qual.ref_median7, qual.rows > qual.cdas],
    ["missing", "partial?", "duplicated rows"], default="ok")
qual_flag = qual[qual.status != "ok"][["snapshot", "rows", "cdas", "ref_median7", "status"]]
save_tab(qual[["snapshot", "rows", "cdas", "value_bn", "ref_median7", "status"]], "tab_debito_quality",
         fmts={"rows": 0, "cdas": 0, "value_bn": 1, "ref_median7": 0},
         caption="DEBITO monthly snapshots, VAT (ICMS) debt of legal entities, 2016-01 to 2025-12")
print(f"{time.time() - t0:.0f} s")
qual_flag

In [ ]:
miss = qual_flag[qual_flag.status == "missing"].snapshot.tolist()
part = qual_flag[qual_flag.status == "partial?"].snapshot.tolist()
dup = qual_flag[qual_flag.status == "duplicated rows"].snapshot.tolist()
reg("qualMissingSnapshots", ", ".join(miss), source="tab_debito_quality")
reg("qualPartialSnapshots", ", ".join(part), source="tab_debito_quality")
reg("qualDuplicatedSnapshots", ", ".join(dup), source="tab_debito_quality")
r24 = qual.set_index("snapshot").loc["2024-12"]
reg("qualDecTwentyFourCdas", int(r24.cdas), 0, "tab_debito_quality 2024-12 cdas")
reg("qualDecTwentyFourRef", int(r24.ref_median7), 0, "tab_debito_quality 2024-12 ref_median7")
r25 = qual.set_index("snapshot").loc["2025-12"]
assert r25.status == "ok", "the December 2025 snapshot must be complete"
print("missing:", miss, "\npartial?:", part, "\nduplicated rows:", dup)

**How to read Figure `fig_debito_quality`.** Horizontal axis: monthly snapshot (2016-01 to 2025-12). Vertical axis: number of distinct VAT (ICMS) CDAs of legal entities in the snapshot (millions). The dotted grey line is the 7-snapshot rolling median used as reference; markers show flagged snapshots (missing snapshots are gaps in the line and an "x" at zero). **What not to conclude:** a drop in a single snapshot is not a change in the portfolio; the 80% rule is a screening threshold fixed in advance, not a test.

In [ ]:
fig = go.Figure()
fig.add_scatter(x=qual.snapshot, y=qual.cdas / 1e6, mode="lines", line=dict(color=CEMEPI["deep_teal"], width=1.5), name="CDAs in snapshot")
fig.add_scatter(x=qual.snapshot, y=qual.ref_median7 / 1e6, mode="lines", line=dict(color=CEMEPI["grey"], dash="dot"), name="rolling median (7)")
for stt, cc, sym in [("partial?", CEMEPI["sand"], "circle"), ("duplicated rows", CEMEPI["green"], "diamond")]:
    d = qual[qual.status == stt]
    fig.add_scatter(x=d.snapshot, y=d.cdas / 1e6, mode="markers", marker=dict(color=cc, size=8, symbol=sym), name=stt)
d = qual[qual.status == "missing"]
fig.add_scatter(x=d.snapshot, y=np.zeros(len(d)), mode="markers", marker=dict(color="#8C4A3A", size=8, symbol="x"), name="missing")
fig.update_layout(yaxis_title="VAT (ICMS) CDAs of legal entities (millions)", showlegend=True,
                  xaxis=dict(tickmode="array", tickvals=[f"{y}-01" for y in range(YEAR_MIN, YEAR_MAX + 1)],
                             ticktext=[str(y) for y in range(YEAR_MIN, YEAR_MAX + 1)]))
save_fig(fig, "fig_debito_quality", qual[["snapshot", "rows", "cdas", "ref_median7", "status"]])

## 4. December 2025 snapshot: bases

**Method.** Read the December 2025 snapshot of all VAT (ICMS) debt types, for all debtors (base for concentration) and with the legal-entity flag (base for profiles). The table gives CDAs and stock by raw type, each type's share of VAT (ICMS) debt and legal-entity coverage (share of CDAs and of stock whose debtor is a legal entity).

In [ ]:
SNAP_SQL = f'''
SELECT ID_DEBITO, TIPO_DEBITO AS tipo, {ORIGIN_SQL} AS origin, VALOR_SEM_HONORARIOS AS value,
       NULLIF(LEFT(REGEXP_REPLACE(COALESCE(CNPJ_DEVEDOR, ''), '[^0-9]', '', 'g'), 8), '') AS root,
       ({PJ_SQL}) AS pj, {DATE_SQL} AS dt_insc,
       SITUACAO_DEBITO AS situation, STATUS_AJUIZAMENTO_DEBITO AS filing
FROM src.DEBITO
WHERE ANO_EXTRACAO = {SNAP[0]} AND MES_EXTRACAO = {SNAP[1]} AND TIPO_DEBITO IN ({TYPES_SQL})
'''
snap_raw = q(SNAP_SQL)
n_rows, n_ids = len(snap_raw), snap_raw.ID_DEBITO.nunique()
snap = (snap_raw.sort_values("value", ascending=False, na_position="last").drop_duplicates("ID_DEBITO")
        .query("value > 0").reset_index(drop=True))
snap["age"] = (pd.Timestamp(SNAP_DATE) - pd.to_datetime(snap.dt_insc)).dt.days / 365.25
del snap_raw
print(f"rows {n_rows:,}; distinct CDAs {n_ids:,}; CDAs with value > 0 {len(snap):,}")

def bases(d: pd.DataFrame, name: str) -> dict:
    return {"type": name, "cdas": len(d), "stock_bn": d.value.sum() / 1e9,
            "pct_cdas_vat": 100 * len(d) / len(snap), "pct_stock_vat": 100 * d.value.sum() / snap.value.sum(),
            "pct_cdas_pj": 100 * d.pj.mean(), "pct_stock_pj": 100 * d.value[d.pj].sum() / d.value.sum()}
tab_bases = pd.DataFrame([bases(snap, "VAT (ICMS), all types")]
                         + [bases(snap[snap.tipo == t], f"{TYPE_EN[t]} ({t})") for t in ICMS_TYPES]
                         + [bases(snap[snap.origin == "assessed"], "Assessed origin (Autuação + Simples Nacional)")])
save_tab(tab_bases, "tab_bases_dec2025", fmts={"cdas": 0, "stock_bn": 1, "pct_cdas_vat": 2, "pct_stock_vat": 2,
                                                "pct_cdas_pj": 2, "pct_stock_pj": 2},
         caption="Bases of the December 2025 snapshot, all debtors; stock in BRL billion")
b = tab_bases.set_index("type")
ALL, ASS, DEC = "VAT (ICMS), all types", "Assessed (ICMS Autuação)", "Declared (ICMS Declarado)"
reg("nCdaVat", b.loc[ALL, "cdas"], 0, "tab_bases_dec2025")
reg("stockVatBn", b.loc[ALL, "stock_bn"], 1, "tab_bases_dec2025")
reg("nCdaAssessed", b.loc[ASS, "cdas"], 0, "tab_bases_dec2025")
reg("stockAssessedBn", b.loc[ASS, "stock_bn"], 1, "tab_bases_dec2025")
reg("nCdaDeclared", b.loc[DEC, "cdas"], 0, "tab_bases_dec2025")
reg("nCdaDeclaredMn", b.loc[DEC, "cdas"] / 1e6, 2, "tab_bases_dec2025")
reg("stockDeclaredBn", b.loc[DEC, "stock_bn"], 1, "tab_bases_dec2025")
reg("pctCdaAssessed", b.loc[ASS, "pct_cdas_vat"], 1, "tab_bases_dec2025")
reg("pctStockAssessed", b.loc[ASS, "pct_stock_vat"], 1, "tab_bases_dec2025")
reg("pctCdaPj", b.loc[ALL, "pct_cdas_pj"], 2, "tab_bases_dec2025")
reg("pctStockPj", b.loc[ALL, "pct_stock_pj"], 2, "tab_bases_dec2025")
tab_bases

## 5. Concentration of the stock

**Method.** For a set of units with values $v_1 \ge v_2 \ge \dots \ge v_n$ and total $V=\sum_i v_i$, the Lorenz curve (Lorenz, 1905) gives the cumulative share of the stock held by the ordered units. Three measures are used:

- share of the largest units: $S(p) = \frac{1}{V}\sum_{i=1}^{m(p)} v_i$, with $m(p)=\max\{1,\operatorname{round}(p\,n)\}$ and $p \in \{0.1\%,\,1\%,\,10\%\}$;
- number of units that add up to half of the stock: $k_{50} = \min\{k : \sum_{i=1}^{k} v_i \ge V/2\}$;
- Gini coefficient, $G = 1 - 2\int_0^1 L(u)\,du$, with $L$ the Lorenz curve in increasing order, integrated by the trapezoidal rule.

**Here.** Unit = CDA (each CDA counts once) or debtor (sum by CNPJ root, only for debtors with a CNPJ; column `pct_stock_with_cnpj` gives how much of the stock enters that level). December 2025 snapshot, all debtors. **What not to conclude:** high concentration says nothing about the chance of recovery, and the debtor- and CDA-level measures answer different questions.

In [ ]:
def lorenz_stats(v: np.ndarray) -> dict:
    v = np.sort(np.asarray(v, dtype=float))[::-1]
    V = v.sum()
    cum = np.cumsum(v) / V
    gini = 1 - 2 * np.trapezoid(np.cumsum(v[::-1]) / V, dx=1 / v.size)
    top = {f"top_{lab}": 100 * v[: max(1, int(round(f * v.size)))].sum() / V
           for lab, f in (("0.1%", 0.001), ("1%", 0.01), ("10%", 0.1))}
    return {"units": v.size, "stock_bn": V / 1e9, "gini": gini, **top,
            "k50": int(np.searchsorted(cum, 0.5)) + 1}

GROUPS = {ALL: snap, ASS: snap[snap.tipo == "ICMS Autuação"], DEC: snap[snap.tipo == "ICMS Declarado"],
          "Assessed origin (Autuação + Simples Nacional)": snap[snap.origin == "assessed"]}
rows_, curves = [], {}
for g, d in GROUPS.items():
    rows_.append({"group": g, "level": "CDA", **lorenz_stats(d.value.values), "pct_stock_with_cnpj": np.nan})
    dv = d[d.root.notna()].groupby("root").value.sum().values
    rows_.append({"group": g, "level": "Debtor (CNPJ root)", **lorenz_stats(dv),
                  "pct_stock_with_cnpj": 100 * d.value[d.root.notna()].sum() / d.value.sum()})
    if g in (ASS, DEC):
        s_ = np.sort(dv)[::-1]
        idx = np.unique(np.geomspace(1, s_.size, 200).round().astype(int))
        curves[g] = pd.DataFrame({"group": g, "debtors": idx, "cum_pct_stock": 100 * np.cumsum(s_)[idx - 1] / s_.sum()})
conc = pd.DataFrame(rows_)
save_tab(conc, "tab_concentration", fmts={"units": 0, "stock_bn": 1, "gini": 3, "top_0.1%": 1, "top_1%": 1,
                                           "top_10%": 1, "k50": 0, "pct_stock_with_cnpj": 1},
         caption="Concentration of the stock, December 2025 snapshot, all debtors")
c = conc.set_index(["group", "level"])
DL = "Debtor (CNPJ root)"
reg("topOneCdaVat", c.loc[(ALL, "CDA"), "top_1%"], 1, "tab_concentration")
reg("kFiftyDebtorVat", c.loc[(ALL, DL), "k50"], 0, "tab_concentration")
reg("kFiftyDebtorAssessed", c.loc[(ASS, DL), "k50"], 0, "tab_concentration")
reg("kFiftyDebtorDeclared", c.loc[(DEC, DL), "k50"], 0, "tab_concentration")
reg("topOneDebtorAssessed", c.loc[(ASS, DL), "top_1%"], 1, "tab_concentration")
reg("topOneDebtorDeclared", c.loc[(DEC, DL), "top_1%"], 1, "tab_concentration")
reg("giniDebtorAssessed", c.loc[(ASS, DL), "gini"], 3, "tab_concentration")
reg("giniDebtorDeclared", c.loc[(DEC, DL), "gini"], 3, "tab_concentration")
conc

**How to read Figure `fig_concentration_origin`.** Panel (a): share of each type's stock held by the largest 0.1%, 1% and 10% of debtors (CNPJ root), side-by-side bars (assessed in deep teal, declared in sand). Panel (b): cumulative curve, number of debtors on a log horizontal axis and cumulative share of the stock on the vertical axis; the dashed line at 50% marks $k_{50}$. **Comparison:** the two raw types against each other. **What not to conclude:** since declared debt has many more debtors, the same fraction (1%) means many more firms in the declared portfolio; differences in $k_{50}$ and in $S(1\%)$ can point in opposite directions.

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=("(a) Share of stock held by the largest debtors",
                                                    "(b) Cumulative stock by number of debtors"), horizontal_spacing=0.12)
fig_rows = []
for g, o in ((ASS, "assessed"), (DEC, "declared")):
    r = c.loc[(g, DL)]
    ys = [r["top_0.1%"], r["top_1%"], r["top_10%"]]
    fig.add_bar(x=["top 0.1%", "top 1%", "top 10%"], y=ys, name=g, marker_color=COL[o],
                text=[fmt(v, 1) for v in ys], textposition="outside", row=1, col=1)
    cv = curves[g]
    fig.add_scatter(x=cv.debtors, y=cv.cum_pct_stock, mode="lines", line=dict(color=COL[o]), showlegend=False, row=1, col=2)
    fig_rows += [{"group": g, "measure": m, "value": v} for m, v in zip(["top_0.1%", "top_1%", "top_10%"], ys)]
    fig_rows += [{"group": g, "measure": "k50", "value": r["k50"]}]
fig.add_hline(y=50, line_dash="dash", line_color=CEMEPI["grey"], row=1, col=2)
fig.update_xaxes(type="log", title_text="debtors (CNPJ root, log scale)", row=1, col=2)
fig.update_yaxes(title_text="% of the type's stock", range=[0, 110], row=1, col=1)
fig.update_yaxes(title_text="cumulative % of stock", range=[0, 102], row=1, col=2)
fig.update_layout(barmode="group", showlegend=True)
save_fig(fig, "fig_concentration_origin", pd.concat([pd.DataFrame(fig_rows), pd.concat(curves.values())]), height=340)

## 6. Profile of the two origins

**Method.** Descriptive statistics by raw type, legal entities only:

- CDA value (mean and median) and age in the snapshot, $\text{age}_i = (t_{\text{snap}} - t^{\text{enrol}}_i)/365.25$ years, with $t_{\text{snap}} =$ 1 Dec 2025 — **December 2025 snapshot**;
- share of CDAs and of stock filed in court (`STATUS_AJUIZAMENTO_DEBITO = 'Ajuizado'`) — **December 2025 snapshot**;
- share of CDAs with any payment and median years from enrolment to first payment — **panel universe**: every legal-entity VAT (ICMS) CDA present in some snapshot from 2016 to 2025, with payments from 2016 to 2025. The December snapshot alone cannot be used here because, by construction, it excludes CDAs already settled.

The next cell completes the CDA × year panel (the year-end stock per CDA was built in section 3), links payments and checks that no 2026 data entered.

In [ ]:
t0 = time.time()
con.execute('''
CREATE OR REPLACE TEMP TABLE attr AS
SELECT ID_DEBITO, arg_max(tipo, CAST(year AS INTEGER) * 100 + last_month) AS tipo, MIN(dt_insc) AS dt_insc,
       arg_max(cnpj, CAST(year AS INTEGER) * 100 + last_month) AS cnpj
FROM ye GROUP BY ID_DEBITO''')
PROG_SQL = " OR ".join(f"CAST(a.DATA_ARRECADACAO_GARE AS DATE) BETWEEN DATE '{i}' AND DATE '{f}'" for _, i, f in PROGRAMMES)
con.execute(f'''
CREATE OR REPLACE TEMP TABLE gare AS
SELECT a.ID_DEBITO, YEAR(a.DATA_ARRECADACAO_GARE) AS year, SUM(a.VALOR_RECEITA) AS paid, COUNT(*) AS n_pay,
       MIN(CAST(a.DATA_ARRECADACAO_GARE AS DATE)) AS first_pay,
       SUM(a.VALOR_RECEITA) FILTER (WHERE {PROG_SQL}) AS paid_prog
FROM src.ARRECADACAO a SEMI JOIN attr t ON a.ID_DEBITO = t.ID_DEBITO
WHERE a.DATA_ARRECADACAO_GARE >= TIMESTAMP '{YEAR_MIN}-01-01' AND a.DATA_ARRECADACAO_GARE < TIMESTAMP '{PAY_END}'
GROUP BY ALL''')
con.execute('''
CREATE OR REPLACE TEMP TABLE panel AS
SELECT y.year, y.ID_DEBITO, y.stock, COALESCE(g.paid, 0) AS paid, a.tipo,
       CASE WHEN a.tipo = 'ICMS Declarado' THEN 'declared' WHEN a.tipo LIKE 'ICMS Autua%' THEN 'assessed' ELSE 'other' END AS origin
FROM ye y LEFT JOIN gare g USING (ID_DEBITO, year) JOIN attr a USING (ID_DEBITO)''')
chk = q("SELECT MIN(year) mn, MAX(year) mx, COUNT(*) n, COUNT(DISTINCT ID_DEBITO) ncda FROM panel").iloc[0]
chk_g = q("SELECT MIN(first_pay) mn, MAX(first_pay) mx FROM gare").iloc[0]
assert int(chk.mx) == YEAR_MAX and str(chk_g.mx) < PAY_END, "2026 data in the panel"
print(f"panel: {int(chk.n):,} CDA-years, {int(chk.ncda):,} CDAs, {int(chk.mn)}–{int(chk.mx)}; "
      f"GARE from {chk_g.mn} to {chk_g.mx}; {time.time() - t0:.0f} s")

In [ ]:
pj = snap[snap.pj].copy()
ticket = pj.groupby("tipo").agg(cdas_pj=("value", "size"), mean_value=("value", "mean"), median_value=("value", "median"),
                                median_age=("age", "median"),
                                pct_cdas_filed=("filing", lambda s: 100 * (s == "Ajuizado").mean()))
pj["v_filed"] = np.where(pj.filing == "Ajuizado", pj.value, 0.0)
ticket["pct_stock_filed"] = 100 * pj.groupby("tipo").v_filed.sum() / pj.groupby("tipo").value.sum()
pay = q('''
SELECT a.tipo, COUNT(*) AS cdas_panel, 100 * AVG(CASE WHEN f.first_pay IS NOT NULL THEN 1 ELSE 0 END) AS pct_any_payment,
       MEDIAN(DATE_DIFF('day', a.dt_insc, f.first_pay) / 365.25) AS median_years_to_first_payment
FROM attr a LEFT JOIN (SELECT ID_DEBITO, MIN(first_pay) AS first_pay FROM gare GROUP BY 1) f USING (ID_DEBITO)
GROUP BY 1''').set_index("tipo")
profile = ticket.join(pay).reindex(ICMS_TYPES)
profile.index = [f"{TYPE_EN[t]} ({t})" for t in profile.index]
profile.index.name = "type"
profile = profile.reset_index()
save_tab(profile, "tab_profile_type", fmts={"cdas_pj": 0, "mean_value": 0, "median_value": 0, "median_age": 2,
                                            "pct_cdas_filed": 1, "pct_stock_filed": 1, "cdas_panel": 0,
                                            "pct_any_payment": 1, "median_years_to_first_payment": 2},
         caption="Profile by raw type, legal entities; value (BRL), age and filing in the Dec 2025 snapshot; payment in the 2016-2025 panel universe")
p = profile.set_index("type")
for k, t in (("Assessed", ASS), ("Declared", DEC)):
    reg(f"meanValue{k}", p.loc[t, "mean_value"], 0, "tab_profile_type")
    reg(f"medianValue{k}", p.loc[t, "median_value"], 0, "tab_profile_type")
    reg(f"medianAge{k}", p.loc[t, "median_age"], 2, "tab_profile_type")
    reg(f"pctCdaFiled{k}", p.loc[t, "pct_cdas_filed"], 1, "tab_profile_type")
    reg(f"pctStockFiled{k}", p.loc[t, "pct_stock_filed"], 1, "tab_profile_type")
    reg(f"pctAnyPayment{k}", p.loc[t, "pct_any_payment"], 1, "tab_profile_type")
    reg(f"yearsFirstPayment{k}", p.loc[t, "median_years_to_first_payment"], 2, "tab_profile_type")
profile

## 7. Tests of differences between types and between origins

**Method.** With millions of CDAs, any difference is statistically significant; decisions rely on **effect sizes**, with thresholds fixed before looking at the data ($|r| \ge 0.1$ as a non-negligible difference; $|r| \ge 0.3$ as substantial).

- **Kruskal–Wallis** (Kruskal and Wallis, 1952) across the five raw types. With $R_j$ the rank sum of group $j$, $n_j$ its size and $n=\sum_j n_j$,
  $$H = \frac{12}{n(n+1)}\sum_{j}\frac{R_j^2}{n_j} - 3(n+1),$$
  corrected for ties (SciPy implementation). Effect size: $\varepsilon^2 = H/(n-1)$.
- **Mann–Whitney** (Mann and Whitney, 1947), assessed against declared. Effect size: **rank-biserial correlation** (Kerby, 2014), $r = 2U_1/(n_1 n_2) - 1$, where $U_1$ is the statistic of the first group; $r>0$ means larger values in the first group. Pairwise $p$-values are adjusted by Benjamini and Hochberg (1995).

**Variables.** CDA value and age in the December 2025 snapshot (legal entities); **recovery rate per CDA** in the 2016–2025 panel, $\text{rate}_i = \sum_t \text{paid}_{it} \,/\, \overline{\text{stock}}_i$, with the mean stock over years with positive stock (same definition as the earlier computation). **Violated assumption:** the tests assume independent CDAs, which fails when one debtor has several CDAs; they therefore describe rather than support inference about debtors. **Alternative considered:** repeat at the debtor level or with resampling by debtor (next step).

**Requested check.** In the earlier computation, the Kruskal–Wallis statistic for the rate was identical to the one in the March 2026 paper ($H \approx 1.46\times10^4$). The hypothesis checked here is that the rate came from a panel that still included 2026; the value recomputed on the panel cut at 2025 is in the comparison table (section 10).

In [ ]:
rate = q('''
SELECT ID_DEBITO, ANY_VALUE(tipo) AS tipo, ANY_VALUE(origin) AS origin, SUM(paid) / AVG(stock) AS rate
FROM panel WHERE stock > 0 GROUP BY ID_DEBITO''')

def kruskal(df, var, grp):
    gs = [g[var].dropna().values for _, g in df.groupby(grp)]
    n = sum(len(g) for g in gs)
    H, pv = stats.kruskal(*gs)
    return {"variable": var, "groups": "five raw types", "n": n, "H": H, "p_value": pv, "epsilon2": H / (n - 1)}

def mwu(a, b):
    a, b = a[~np.isnan(a)], b[~np.isnan(b)]
    U, pv = stats.mannwhitneyu(a, b, alternative="two-sided", method="asymptotic")
    return U, pv, 2 * U / (len(a) * len(b)) - 1, len(a), len(b)

frames = {"value": pj[["tipo", "origin", "value"]], "age": pj[["tipo", "origin", "age"]],
          "rate": rate[["tipo", "origin", "rate"]]}
kw = pd.DataFrame([kruskal(f, v, "tipo") for v, f in frames.items()])
pairs = []
for v, f in frames.items():
    for name, ga, gb, col in (("Assessed vs Declared (raw types)", "ICMS Autuação", "ICMS Declarado", "tipo"),
                              ("assessed vs declared (origin)", "assessed", "declared", "origin")):
        U, pv, r, na, nb = mwu(f.loc[f[col] == ga, v].values.astype(float), f.loc[f[col] == gb, v].values.astype(float))
        pairs.append({"variable": v, "comparison": name, "n_a": na, "n_b": nb, "U": U, "p_value": pv, "rank_biserial": r})
pairs = pd.DataFrame(pairs)
pairs["p_bh"] = stats.false_discovery_control(pairs.p_value.clip(lower=1e-300), method="bh")
save_tab(kw, "tab_kruskal_types", fmts={"n": 0, "H": 0, "epsilon2": 4}, caption="Kruskal-Wallis across the five raw types")
save_tab(pairs, "tab_mannwhitney_origin", fmts={"n_a": 0, "n_b": 0, "U": 0, "rank_biserial": 3},
         caption="Mann-Whitney assessed vs declared, rank-biserial correlation")
pr = pairs.set_index(["variable", "comparison"])
for v, k in (("value", "Value"), ("age", "Age"), ("rate", "Rate")):
    reg(f"rBis{k}", abs(pr.loc[(v, "assessed vs declared (origin)"), "rank_biserial"]), 3, "tab_mannwhitney_origin |r| origin")
    reg(f"rBisType{k}", abs(pr.loc[(v, "Assessed vs Declared (raw types)"), "rank_biserial"]), 3, "tab_mannwhitney_origin |r| raw types")
reg("kwRateH", kw.set_index("variable").loc["rate", "H"], 0, "tab_kruskal_types")
display(kw)
pairs

## 8. Stock, payment and recovery by year (2016–2025)

**Method.** In the CDA × year panel, for each origin $o$ and year $t$: stock $E_{ot}=\sum_{i \in o}\text{stock}_{it}$, paid $P_{ot}=\sum_{i\in o}\text{paid}_{it}$ and the **annual recovery rate** $\rho_{ot} = 100\,P_{ot}/E_{ot}$ (paid in the year over year-end stock, in %). It is a ratio of aggregates, not the mean of CDA rates. **Limitations:** years with missing or partial snapshots (section 3) change the snapshot that defines the year's stock; payments of a CDA that is in no snapshot of the year are not counted.

In [ ]:
series = q('''
SELECT year, origin, COUNT(*) AS cdas, SUM(stock) / 1e9 AS stock_bn, SUM(paid) / 1e9 AS paid_bn
FROM panel GROUP BY ALL ORDER BY origin, year''')
series["rate_pct"] = 100 * series.paid_bn / series.stock_bn
tot = series[series.origin != "other"].groupby("year").paid_bn.sum()
series = series.merge((100 * series[series.origin == "declared"].set_index("year").paid_bn / tot)
                      .rename("pct_declared_in_paid").reset_index(), on="year", how="left")
save_tab(series, "tab_annual_series_origin", fmts={"cdas": 0, "stock_bn": 1, "paid_bn": 3, "rate_pct": 3, "pct_declared_in_paid": 1},
         caption="CDA x year panel, VAT (ICMS) debt of legal entities, 2016-2025, by origin; BRL billion")
s = series.set_index(["origin", "year"])
for o, k in (("declared", "Declared"), ("assessed", "Assessed")):
    d = series[series.origin == o]
    reg(f"rateMin{k}", d.rate_pct.min(), 2, "tab_annual_series_origin")
    reg(f"rateMinYear{k}", str(int(d.loc[d.rate_pct.idxmin(), "year"])), source="tab_annual_series_origin")
    reg(f"rateMax{k}", d.rate_pct.max(), 2, "tab_annual_series_origin")
    reg(f"rateMaxYear{k}", str(int(d.loc[d.rate_pct.idxmax(), "year"])), source="tab_annual_series_origin")
    reg(f"rateTwentyFive{k}", s.loc[(o, 2025), "rate_pct"], 2, "tab_annual_series_origin 2025")
reg("cdaAssessedNineteen", s.loc[("assessed", 2019), "cdas"], 0, "tab_annual_series_origin")
reg("cdaAssessedTwenty", s.loc[("assessed", 2020), "cdas"], 0, "tab_annual_series_origin")
reg("paidAssessedNineteen", s.loc[("assessed", 2019), "paid_bn"], 2, "tab_annual_series_origin")
reg("paidAssessedTwentyFour", s.loc[("assessed", 2024), "paid_bn"], 2, "tab_annual_series_origin")
series[series.origin != "other"]

**How to read Figure `fig_annual_series_origin`.** Panel (a): year-end stock by origin (BRL billion), 2016–2025. Panel (b): annual recovery rate $\rho_{ot}$ in %, on a log scale so that both origins fit the same axis. Cream bands mark the years with missing snapshots (2018 and 2020) and the year of the partial December snapshot (2024). **What not to conclude:** a jump in an isolated year is not a trend, and the log scale makes small differences at low levels look large.

### 8.1 Open research questions on 2019, 2020 and 2024 (no cause asserted)

The assessed series has three features this notebook **does not explain**: the fall in the number of assessed CDAs in the panel between 2019 and 2020, and the spikes in assessed payments in 2019 and 2024. They are registered as research questions, to be investigated and re-run once the `DEBITO` gaps are restored:

- **RQ-A (2020 fall).** Is the fall an artefact of the missing October–December 2020 snapshots (the 2020 stock then comes from an earlier snapshot, and CDAs settled or cancelled before it drop out), or a real change in the assessed portfolio?
- **RQ-B (2019 and 2024 spikes).** Are the payment spikes associated with instalment programmes (*parcelamento*) or settlement agreements (*transação*) in those years — for example, the windows labelled PEP 2019 and Acordo Paulista 2024 used below — or with a few very large payments? The test needs payment-level timing and the programme rules, not only annual totals.
- **RQ-C (data).** Do the conclusions on recovery by origin survive re-running the panel on a `DEBITO` table without the missing and partial snapshots?

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=("(a) Year-end stock (BRL bn)", "(b) Paid in year / stock (%, log)"),
                    horizontal_spacing=0.12)
for o in ("assessed", "declared"):
    d = series[series.origin == o]
    fig.add_scatter(x=d.year, y=d.stock_bn, mode="lines+markers", name=o, line=dict(color=COL[o]), row=1, col=1)
    fig.add_scatter(x=d.year, y=d.rate_pct, mode="lines+markers", name=o, line=dict(color=COL[o]), showlegend=False, row=1, col=2)
for a in (2018, 2020, 2024):
    for cc in (1, 2):
        fig.add_vrect(x0=a - 0.4, x1=a + 0.4, fillcolor=CEMEPI["cream"], line_width=0, layer="below", row=1, col=cc)
fig.update_yaxes(type="log", row=1, col=2)
fig.update_xaxes(dtick=1)
fig.update_layout(showlegend=True)
save_fig(fig, "fig_annual_series_origin", series[series.origin != "other"], height=340)

## 9. The largest debts: composition and clustering

### 9.1 Composition of the top

**Method.** Legal-entity VAT (ICMS) CDAs in the December 2025 snapshot are ranked by value; the top 1% has $\operatorname{round}(0.01\,n)$ CDAs and the top 0.1% has $\operatorname{round}(0.001\,n)$. For each cut, the table gives each origin’s share of CDAs and of value. **The rest of the portfolio is what lies outside the top 1%** (author’s decision, 27 Oct 2026); the top 0.1% is shown only as a finer cut inside the top 1%, with no “rest” of its own.

In [ ]:
rk = pj.sort_values("value", ascending=False).reset_index(drop=True)
k_tenth, k_one = max(1, round(0.001 * len(rk))), max(1, round(0.01 * len(rk)))
# the "rest" of the portfolio is what lies outside the top 1% (author's decision, 27 Oct 2026)
parts = (("top 0.1%", rk.iloc[:k_tenth]), ("top 1%", rk.iloc[:k_one]), ("rest (outside top 1%)", rk.iloc[k_one:]))
comp = []
for part_, d in parts:
    for o in ("assessed", "declared", "other"):
        comp.append({"cut": part_, "origin": o, "cdas": int((d.origin == o).sum()),
                     "pct_cdas": 100 * (d.origin == o).mean(), "pct_value": 100 * d.value[d.origin == o].sum() / d.value.sum()})
comp = pd.DataFrame(comp)
save_tab(comp, "tab_top_composition", fmts={"cdas": 0, "pct_cdas": 2, "pct_value": 2},
         caption="Composition by origin of the largest legal-entity VAT (ICMS) CDAs, Dec 2025; rest = outside the top 1%")
cp = comp.set_index(["cut", "origin"])
reg("topOnePctCdaAssessed", cp.loc[("top 1%", "assessed"), "pct_cdas"], 1, "tab_top_composition")
reg("topOnePctValueAssessed", cp.loc[("top 1%", "assessed"), "pct_value"], 1, "tab_top_composition")
reg("topTenthPctCdaAssessed", cp.loc[("top 0.1%", "assessed"), "pct_cdas"], 1, "tab_top_composition")
reg("topTenthPctValueAssessed", cp.loc[("top 0.1%", "assessed"), "pct_value"], 1, "tab_top_composition")
reg("restOnePctCdaAssessed", cp.loc[("rest (outside top 1%)", "assessed"), "pct_cdas"], 1, "tab_top_composition")
reg("restOnePctValueAssessed", cp.loc[("rest (outside top 1%)", "assessed"), "pct_value"], 1, "tab_top_composition")
reg("topOneNCda", k_one, 0, "tab_top_composition")
comp

### 9.2 K-means on the top 1% of CDAs ($k=4$, descriptive)

**Method.** K-means (Lloyd, 1982) minimises the within-cluster sum of squares,
$$\min_{C_1,\dots,C_k}\sum_{j=1}^{k}\sum_{x_i\in C_j}\lVert x_i-\mu_j\rVert^2 .$$
For each $k \in \{2,\dots,8\}$, 100 fits with k-means++ initialisation and distinct seeds (`SEED + 1000k + s`) are run and the lowest-inertia partition is kept. Quality is measured by the **silhouette** (Rousseeuw, 1987), $s_i = (b_i-a_i)/\max(a_i,b_i)$, with $a_i$ the mean distance from $i$ to its own cluster and $b_i$ the smallest mean distance to another cluster, computed on a fixed subsample of 5,000 CDAs with two distances: Euclidean on the K-means matrix and **Gower** (Gower, 1971), $d_{ij}=\frac{1}{p}\sum_{f=1}^{p}\delta^{(f)}_{ij}$, with $\delta^{(f)}_{ij}=|x_{if}-x_{jf}|/R_f$ for numeric variables ($R_f$ the range) and $\delta^{(f)}_{ij}=\mathbb{1}[x_{if}\neq x_{jf}]$ for categorical ones.

**Inputs (9 variables).** Numeric: $\log_{10}$ value, age, cumulative recovery (paid/value), years to first payment (age if never paid), number of payments and share of payments made inside programme windows (PEP 2019 and Acordo Paulista 2024; windows in `PROGRAMMES`). Categorical: **origin** (`ICMS_ORIGIN`), situation and court filing status. K-means matrix: numeric variables rescaled to $[0,1]$ by their range (nulls replaced by the median) and categorical variables one-hot encoded and multiplied by $1/\sqrt{2}$, so that each variable contributes at most 1 to the squared distance. For Gower, numeric variables are standardised by median and interquartile range before the range.

**Caution: origin is one of the inputs.** A separation of the clusters by origin is partly a consequence of the construction; section 9.3 therefore repeats the clustering **without** origin. **$k=4$ is the author's decision**, taken before this notebook; the silhouette-by-$k$ table only accompanies it. **[ÉTORE DECIDE]** The programme windows are inherited from the earlier computation and must be confirmed.

In [ ]:
K = max(1, round(0.01 * len(rk)))
top = rk.iloc[:K].copy()
con.register("top_ids", pd.DataFrame({"ID_DEBITO": top.ID_DEBITO.astype("int64").values}))
gtop = q('''
SELECT g.ID_DEBITO, MIN(g.first_pay) AS first_pay, SUM(g.n_pay) AS n_pay, SUM(g.paid) AS paid_total,
       SUM(COALESCE(g.paid_prog, 0)) AS paid_prog
FROM gare g SEMI JOIN top_ids t ON g.ID_DEBITO = t.ID_DEBITO GROUP BY 1''')
at = q("SELECT a.ID_DEBITO, a.dt_insc AS dt_attr FROM attr a SEMI JOIN top_ids t ON a.ID_DEBITO = t.ID_DEBITO")
con.unregister("top_ids")
top = top.merge(gtop, on="ID_DEBITO", how="left").merge(at, on="ID_DEBITO", how="left")
top[["n_pay", "paid_total", "paid_prog"]] = top[["n_pay", "paid_total", "paid_prog"]].fillna(0)
top["log_value"] = np.log10(top.value)
top["recovery"] = top.paid_total / top.value
top["programme_share"] = np.where(top.paid_total > 0, top.paid_prog / top.paid_total.where(top.paid_total > 0), 0.0)
top["time_to_payment"] = ((pd.to_datetime(top.first_pay) - pd.to_datetime(top.dt_attr)).dt.days / 365.25).fillna(top.age)
top["n_payments"] = top.n_pay
top = top.sort_values("ID_DEBITO").reset_index(drop=True)
NUM = ["log_value", "age", "recovery", "time_to_payment", "n_payments", "programme_share"]
CAT = ["origin", "situation", "filing"]

reg("kmN", K, 0, "top 1% legal entities Dec 2025")
reg("kmMinValueMn", top.value.min() / 1e6, 2, "top 1% legal entities Dec 2025")
reg("kmSumBn", top.value.sum() / 1e9, 1, "top 1% legal entities Dec 2025")
reg("kmPctPaid", 100 * (top.n_pay > 0).mean(), 1, "top 1% legal entities Dec 2025")

def design(df, cat):
    X = df[NUM].to_numpy(float)
    X = np.where(np.isnan(X), np.nanmedian(X, axis=0), X)
    rng = X.max(axis=0) - X.min(axis=0); rng[rng == 0] = 1
    blocks = [(X - X.min(axis=0)) / rng]
    for c_ in cat:
        codes = df[c_].astype(str).to_numpy()
        blocks.append((codes[:, None] == np.unique(codes)[None, :]).astype(float) / np.sqrt(2))
    return np.hstack(blocks)

def gower_sub(df, cat, sub):
    X = df[NUM].to_numpy(float)
    med, iqr = np.nanmedian(X, axis=0), np.subtract(*np.nanpercentile(X, [75, 25], axis=0))
    Xn = np.nan_to_num((X - med) / np.where(iqr == 0, 1, iqr))
    rng = Xn.max(axis=0) - Xn.min(axis=0); rng[rng == 0] = 1
    A = (Xn / rng)[sub]
    Cc = np.column_stack([pd.factorize(df[c_].astype(str))[0] for c_ in cat])[sub]
    D = np.zeros((len(sub), len(sub)))
    for j in range(A.shape[1]):
        D += np.abs(A[:, j][:, None] - A[:, j][None, :])
    for j in range(Cc.shape[1]):
        D += Cc[:, j][:, None] != Cc[:, j][None, :]
    return D / (len(NUM) + len(cat))

def multi_init(X, k, n_seeds=100):
    best = None
    for s_ in range(n_seeds):
        km = KMeans(n_clusters=k, init="k-means++", n_init=1, max_iter=500, random_state=SEED + 1000 * k + s_).fit(X)
        if best is None or km.inertia_ < best.inertia_:
            best = km
    return best

SUB = np.sort(np.random.default_rng(SEED).choice(len(top), size=min(5000, len(top)), replace=False))
X_km = design(top, CAT)
D_sub = gower_sub(top, CAT, SUB)
t0 = time.time()
sel, fits = [], {}
for k in range(2, 9):
    km = multi_init(X_km, k)
    fits[k] = km.labels_
    sizes = np.bincount(km.labels_, minlength=k)
    sel.append({"k": k, "inertia": km.inertia_, "silhouette_euclid": silhouette_score(X_km[SUB], km.labels_[SUB]),
                "silhouette_gower": silhouette_score(D_sub, km.labels_[SUB], metric="precomputed"),
                "smallest_cluster_pct": 100 * sizes.min() / sizes.sum()})
sel = pd.DataFrame(sel)
save_tab(sel, "tab_kmeans_selection", fmts={"inertia": 1, "silhouette_euclid": 3, "silhouette_gower": 3, "smallest_cluster_pct": 1},
         caption="K-means on the top 1%: silhouette by k (5,000 subsample), 100 initialisations per k")
print(f"{time.time() - t0:.0f} s")
sel

In [ ]:
K_STAR = 4   # author's decision (not chosen by this notebook)
lab4 = fits[K_STAR]
order = pd.Series(lab4).value_counts().index.tolist()         # G1 = largest cluster
top["cluster"] = pd.Series(lab4).map({c_: f"G{i + 1}" for i, c_ in enumerate(order)}).values
mode = lambda s: s.value_counts().idxmax()
prof = top.groupby("cluster").agg(cdas=("value", "size"), value_bn=("value", lambda v: v.sum() / 1e9),
                                  origin_mode=("origin", mode),
                                  pct_origin_mode=("origin", lambda s: 100 * s.value_counts(normalize=True).iloc[0]),
                                  situation_mode=("situation", mode), filing_mode=("filing", mode),
                                  median_value_mn=("value", lambda v: v.median() / 1e6), median_age=("age", "median"),
                                  pct_paid=("n_pay", lambda s: 100 * (s > 0).mean())).reset_index()
prof["pct_cdas"] = 100 * prof.cdas / prof.cdas.sum()
prof["pct_value"] = 100 * prof.value_bn / prof.value_bn.sum()
save_tab(prof, "tab_kmeans_k4_profiles", fmts={"cdas": 0, "value_bn": 1, "pct_origin_mode": 1, "median_value_mn": 2,
                                                "median_age": 2, "pct_paid": 1, "pct_cdas": 1, "pct_value": 1},
         caption="K-means k=4 clusters on the top 1% (G1 = largest); situation and filing modes are raw database labels")
sk = sel.set_index("k")
reg("kmSilEuclid", sk.loc[K_STAR, "silhouette_euclid"], 3, "tab_kmeans_selection k=4")
reg("kmSilGower", sk.loc[K_STAR, "silhouette_gower"], 3, "tab_kmeans_selection k=4")
reg("kmBestKEuclid", str(int(sk.silhouette_euclid.idxmax())), source="tab_kmeans_selection")
reg("kmBestKGower", str(int(sk.silhouette_gower.idxmax())), source="tab_kmeans_selection")
for _, r in prof.iterrows():
    gname = {"G1": "One", "G2": "Two", "G3": "Three", "G4": "Four"}[r.cluster]
    reg(f"kmG{gname}PctCda", r.pct_cdas, 1, "tab_kmeans_k4_profiles")
    reg(f"kmG{gname}PctValue", r.pct_value, 1, "tab_kmeans_k4_profiles")

pca = PCA(n_components=3, random_state=SEED).fit(X_km)
names = NUM + [f"{c_}={v}" for c_ in CAT for v in np.unique(top[c_].astype(str))]
load = pd.DataFrame(pca.components_.T, index=names, columns=["PC1", "PC2", "PC3"])
pca_tab = pd.DataFrame({"component": ["PC1", "PC2", "PC3"], "pct_variance": 100 * pca.explained_variance_ratio_,
                        "largest_loading_on": [load[c_].abs().idxmax() for c_ in load.columns],
                        "loading": [load[c_][load[c_].abs().idxmax()] for c_ in load.columns]})
save_tab(pca_tab, "tab_kmeans_pca", fmts={"pct_variance": 1, "loading": 2}, caption="PCA of the K-means matrix")
reg("pcaOnePct", pca_tab.pct_variance.iloc[0], 1, "tab_kmeans_pca")
display(prof); display(pca_tab)
load.round(2)

### 9.3 Sensitivity: K-means without origin

**Method.** Same procedure ($k=4$, 100 initialisations) with the categorical inputs reduced to situation and filing status. It is compared with the main solution by the **adjusted Rand index** (ARI; 1 = same partition, 0 = agreement expected by chance) and by origin purity (share of CDAs that belong to the most frequent origin of their cluster, weighted over clusters); the reference is the purity of a single cluster (the overall share of the most frequent origin). **Reading:** if, without origin, the clusters remain almost pure in origin, the separation is not only an artefact of the construction; if purity falls towards the reference, origin was doing the work.

In [ ]:
X_no = design(top, ["situation", "filing"])
km_no = multi_init(X_no, K_STAR)
top["cluster_no_origin"] = km_no.labels_
pur = (top.groupby("cluster_no_origin").origin.agg(lambda s: 100 * s.value_counts(normalize=True).iloc[0]).rename("origin_purity")
       .to_frame().join(top.groupby("cluster_no_origin").origin.agg(mode).rename("origin_mode"))
       .join(top.groupby("cluster_no_origin").size().rename("cdas")).reset_index())
pur["pct_cdas"] = 100 * pur.cdas / pur.cdas.sum()
ari = adjusted_rand_score(lab4, km_no.labels_)
sil_no = silhouette_score(X_no[SUB], km_no.labels_[SUB])
pur_main = 100 * top.groupby("cluster").origin.agg(lambda s: s.value_counts().iloc[0]).sum() / len(top)
pur_no = 100 * top.groupby("cluster_no_origin").origin.agg(lambda s: s.value_counts().iloc[0]).sum() / len(top)
base_pur = 100 * top.origin.value_counts(normalize=True).iloc[0]
sens = pd.DataFrame([
    {"solution": "with origin (main)", "ari_vs_main": 1.0, "silhouette_euclid": sk.loc[K_STAR, "silhouette_euclid"], "origin_purity": pur_main},
    {"solution": "without origin", "ari_vs_main": ari, "silhouette_euclid": sil_no, "origin_purity": pur_no},
    {"solution": "reference: single cluster", "ari_vs_main": np.nan, "silhouette_euclid": np.nan, "origin_purity": base_pur}])
save_tab(sens, "tab_kmeans_without_origin", fmts={"ari_vs_main": 3, "silhouette_euclid": 3, "origin_purity": 1},
         caption="Sensitivity of K-means k=4 to dropping origin")
reg("kmNoOriginAri", ari, 3, "tab_kmeans_without_origin")
reg("kmPurityWith", pur_main, 1, "tab_kmeans_without_origin")
reg("kmPurityWithout", pur_no, 1, "tab_kmeans_without_origin")
reg("kmPurityBase", base_pur, 1, "tab_kmeans_without_origin")
display(sens)
pur

**How to read Figure `fig_kmeans_k4_top1`.** Two stacked horizontal bars for the four clusters of the main solution: the lower bar splits the CDAs of the top 1%, the upper bar splits their value. Colour = cluster; the legend gives the cluster and its most frequent origin. **What not to conclude:** origin is an input of the clustering, so "single-origin" clusters do not by themselves show that origin is the natural axis of the largest debts (see section 9.3); small clusters may be unstable across seeds.

In [ ]:
colours = [CEMEPI["deep_teal"], CEMEPI["sand"], CEMEPI["teal"], CEMEPI["green"]]
fig = go.Figure()
for i, r in prof.iterrows():
    fig.add_bar(y=["% of value", "% of CDAs"], x=[r.pct_value, r.pct_cdas], orientation="h", marker_color=colours[i],
                name=f"{r.cluster} · {r.origin_mode}", text=[r.cluster if r.pct_value > 4 else "", r.cluster if r.pct_cdas > 4 else ""],
                textposition="inside", insidetextanchor="middle")
fig.update_layout(barmode="stack", showlegend=True, xaxis=dict(range=[0, 100], title="%"),
                  title=dict(text=f"K-means (k = 4) on the top 1% of legal-entity VAT (ICMS) CDAs, Dec 2025 (n = {fmt(K, 0)})", x=0, font=dict(size=12)))
save_fig(fig, "fig_kmeans_k4_top1", prof[["cluster", "origin_mode", "pct_cdas", "pct_value"]], height=260)

## 10. Debtor firm characteristics

This section adds two firm-level characteristics of the legal-entity debtors of VAT (ICMS), taken from the CEMEPI enrichment of the Federal Revenue (RFB) open CNPJ register, `cnpj_enriquecido` (version `rfb-2026-03`, manifest `data_manifests/cnpj_enrich/cnpj_enriquecido_2026-03.json`): (1) the **registration status** of the CNPJ (active, unfit, suspended, closed, null) and (2) the **economic activity** at CNAE **division** level (2 digits). Both are crossed with origin (assessed vs declared) and with recovery.

**Data and join.** `cnpj_enriquecido` has one row per debtor CNPJ (14 characters). It is read **read-only** from its Parquet files, with only the columns used here (CNPJ as join key, match level, CNAE division and section, registration status); firm names, addresses and partners are not in it and are not read. The join key is the debtor's 14-character CNPJ in `DEBITO`, normalised as in the enrichment pipeline (upper case, punctuation removed, 12–13 digit numbers left-padded with zeros). `CPF_DEVEDOR` and `NOME_DEVEDOR` are not read anywhere in this notebook. The CNPJ stays inside DuckDB temporary tables; only aggregates leave.

**Descriptors only (author’s decision, 27 Oct 2026).** Registration status and CNAE division enter the article only as descriptors of the debtors in each origin. No statement in this section says that a status or an activity raises or lowers recovery, or that origin is explained by them.

**Timing caveat.** The register is the March 2026 RFB release: registration status and CNAE are the firm's attributes **as of March 2026**, observed after the debt and payment outcomes (which stop in December 2025). They describe the debtor; they are not a cause measured before the outcome. A firm closed or declared unfit may have reached that status *because* of its debts. No 2026 debt or payment data enter.

**Pinned version.** The cell below checks the row count and the SHA-256 of every Parquet file against the manifest and stops if they differ.

In [ ]:
import hashlib
ENRICH_REF = "2026-03"
ENRICH_DIR = (Path(os.environ["CNPJ_PIPELINE_WORKDIR"]) if os.environ.get("CNPJ_PIPELINE_WORKDIR")
              else DB.parent.parent / "cnpj_enrich") / "build" / ENRICH_REF / "cnpj_enriquecido"
man = json.loads((REPO / "data_manifests/cnpj_enrich" / f"cnpj_enriquecido_{ENRICH_REF}.json").read_text())
for f in man["arquivos"]:
    h = hashlib.sha256((ENRICH_DIR / f["arquivo"]).read_bytes()).hexdigest()
    assert h == f["sha256"], f"cnpj_enriquecido changed: {f['arquivo']}"
glob = str(ENRICH_DIR / "*.parquet")
n_enr = con.execute(f"SELECT COUNT(*), COUNT(DISTINCT cnpj) FROM read_parquet('{glob}')").fetchone()
assert n_enr[0] == man["linhas"] == n_enr[1], n_enr
con.execute(f'''
CREATE OR REPLACE TEMP TABLE firm AS
SELECT cnpj, match_nivel, cnae_divisao, cnae_secao, situacao_cadastral
FROM read_parquet('{glob}')''')
ENV["cnpj_enriquecido"] = {"version": man["versao"], "rows": n_enr[0]}
print(f"cnpj_enriquecido {man['versao']}: {n_enr[0]:,} CNPJs, sha256 checked for {len(man['arquivos'])} files")

### 10.0 Units, universes and coverage

- **Unit.** The registration status belongs to the establishment (14-character CNPJ) and the CNAE to its main activity; both analyses therefore count debtors as **distinct CNPJs** (not CNPJ roots).
- **Stock universe:** legal-entity VAT (ICMS) CDAs in the December 2025 snapshot (year 2025 of the panel, December row), stock $>0$.
- **Recovery universe:** the CDA × year panel of section 6 (2016–2025), with the **pooled recovery rate** $\bar\rho_g = 100\,\sum_t P_{gt} / \sum_t E_{gt}$ for group $g$: payments in all years over year-end stock in all years. It is the multi-year version of the annual rate of section 8 and has the same limitations (missing and partial snapshots).
- **Coverage** is the share of CDAs and of stock whose CNPJ is found in the register: at establishment level (status and CNAE known), at root level only (firm found, establishment not found: status and CNAE unknown) or not found. Unmatched CDAs are kept as their own category, never dropped silently.

In [ ]:
STATUS_EN = {"02": "active", "04": "unfit (inapta)", "03": "suspended", "08": "closed (baixada)", "01": "null (nula)"}
STATUS_SQL = '''CASE WHEN f.cnpj IS NULL OR f.match_nivel = 'nao_encontrado' THEN 'not found in register'
                     WHEN f.match_nivel = 'raiz' THEN 'root only (status unknown)'
                     ELSE COALESCE(MAP {'02': 'active', '04': 'unfit (inapta)', '03': 'suspended',
                                        '08': 'closed (baixada)', '01': 'null (nula)'}[f.situacao_cadastral],
                                   'other code') END'''
STATUS_ORDER = ["active", "unfit (inapta)", "suspended", "closed (baixada)", "null (nula)", "other code",
                "root only (status unknown)", "not found in register"]
t0 = time.time()
con.execute(f'''
CREATE OR REPLACE TEMP TABLE cda_firm AS
SELECT a.ID_DEBITO, a.cnpj, a.dt_insc,
       CASE WHEN a.tipo = 'ICMS Declarado' THEN 'declared' WHEN a.tipo LIKE 'ICMS Autua%' THEN 'assessed' ELSE 'other' END AS origin,
       COALESCE(f.match_nivel, 'nao_encontrado') AS match_level, {STATUS_SQL} AS status,
       CASE WHEN f.match_nivel = 'estabelecimento' THEN f.cnae_divisao END AS division,
       CASE WHEN f.match_nivel = 'estabelecimento' THEN f.cnae_secao END AS section
FROM attr a LEFT JOIN firm f ON f.cnpj = a.cnpj''')
con.execute('''
CREATE OR REPLACE TEMP TABLE pf AS
SELECT p.year, p.ID_DEBITO, p.stock, p.paid, c.cnpj, c.origin, c.status, c.division, c.section, c.match_level,
       (p.year = 2025 AND y.last_month = 12 AND p.stock > 0) AS in_dec2025
FROM panel p JOIN cda_firm c USING (ID_DEBITO) JOIN ye y USING (ID_DEBITO, year)
WHERE c.origin <> 'other' ''')
cov = q('''
SELECT origin, match_level, COUNT(*) FILTER (WHERE in_dec2025) AS cdas_dec2025,
       SUM(stock) FILTER (WHERE in_dec2025) / 1e9 AS stock_dec2025_bn,
       COUNT(DISTINCT cnpj) FILTER (WHERE in_dec2025) AS cnpjs_dec2025,
       COUNT(DISTINCT ID_DEBITO) AS cdas_panel
FROM pf GROUP BY ALL ORDER BY origin, match_level''')
for c in ("cdas_dec2025", "stock_dec2025_bn", "cdas_panel"):
    cov[f"pct_{c}"] = 100 * cov[c] / cov.groupby("origin")[c].transform("sum")
save_tab(cov, "tab_firm_coverage", fmts={"cdas_dec2025": 0, "stock_dec2025_bn": 1, "cnpjs_dec2025": 0, "cdas_panel": 0,
                                         "pct_cdas_dec2025": 1, "pct_stock_dec2025_bn": 1, "pct_cdas_panel": 1},
         caption="Coverage of the join with cnpj_enriquecido (rfb-2026-03), legal-entity VAT (ICMS) CDAs, by origin")
cv = q('''
SELECT COUNT(*) FILTER (WHERE in_dec2025) AS n, COUNT(*) FILTER (WHERE in_dec2025 AND match_level = 'estabelecimento') AS m,
       SUM(stock) FILTER (WHERE in_dec2025) AS s, SUM(stock) FILTER (WHERE in_dec2025 AND match_level = 'estabelecimento') AS sm,
       COUNT(DISTINCT cnpj) FILTER (WHERE in_dec2025) AS nc,
       COUNT(DISTINCT ID_DEBITO) AS npan, COUNT(DISTINCT ID_DEBITO) FILTER (WHERE match_level = 'estabelecimento') AS mpan
FROM pf''').iloc[0]
reg("firmNCnpjDec", cv.nc, 0, "tab_firm_coverage")
reg("firmCovCdaDec", 100 * cv.m / cv.n, 1, "tab_firm_coverage")
reg("firmCovStockDec", 100 * cv.sm / cv.s, 1, "tab_firm_coverage")
reg("firmNCdaPanel", cv.npan, 0, "tab_firm_coverage")
reg("firmCovCdaPanel", 100 * cv.mpan / cv.npan, 1, "tab_firm_coverage")
print(f"{time.time() - t0:.0f} s")
cov

### 10.1 Registration status of the CNPJ and recovery, by origin

**Method.** For each origin and registration status (March 2026 register):

- distinct CNPJs, CDAs and stock in the December 2025 snapshot, and the status's share of the origin's stock;
- the pooled recovery rate $\bar\rho_g$ in the 2016–2025 panel and the share of panel CDAs with any payment;
- **Kaplan–Meier** curves of the time from enrolment to the **first payment** (Kaplan & Meier, 1958). Duration $T_i$ = months from enrolment to the first GARE payment; CDAs without payment by 31 Dec 2025 are right-censored at that date. The estimator is $\hat S(t)=\prod_{t_j\le t}\left(1-d_j/n_j\right)$, with $d_j$ first payments in month $t_j$ and $n_j$ CDAs still at risk; the curves show $1-\hat S(t)$, the share that has made a first payment by month $t$. To avoid left truncation (payments before 2016 are not in the panel), the curves use only the **enrolment cohort 2016–2025**; a status absent from the Kaplan–Meier table has no CDA in that cohort (in this extraction, the CDAs whose CNPJ is not found in the register, or only at root level, were all enrolled before 2016). CDAs settled before ever appearing in a monthly snapshot are not in the panel, which biases $1-\hat S(t)$ downwards in the first months.

**Disclosure control.** Any status × origin cell with fewer than 5 distinct CNPJs is pooled into `other (suppressed)`.

**Limitations.** Descriptive only: no test is reported, because with millions of CDAs every difference is significant and the status is measured after the outcome (see the timing caveat). Censoring is assumed independent of the payment process within each group, which is a strong assumption for closed or unfit firms.

In [ ]:
def suppress(df, key, n_col, label="other (suppressed)", k=5):
    # pool rows whose count of distinct CNPJs in n_col is 1-4 (0 is not disclosive) into one row per origin
    small = (df[n_col] > 0) & (df[n_col] < k)
    out = df.copy()
    out.loc[small, key] = label
    return out, int(small.sum())

def blank_small_pool(df, key, n_cols, label="other (suppressed)", k=5):
    # a pooled row that still has fewer than k CNPJs is blanked (all numbers set to missing)
    m = (df[key] == label) & (df[n_cols].replace(0, np.nan).min(axis=1) < k)
    num_cols = df.select_dtypes("number").columns
    df.loc[m, num_cols] = np.nan
    return df

st_raw = q('''
SELECT origin, status,
       COUNT(DISTINCT cnpj) FILTER (WHERE in_dec2025) AS cnpjs_dec2025,
       COUNT(*) FILTER (WHERE in_dec2025) AS cdas_dec2025,
       COALESCE(SUM(stock) FILTER (WHERE in_dec2025), 0) AS stock_dec2025,
       COUNT(DISTINCT cnpj) AS cnpjs_panel, COUNT(DISTINCT ID_DEBITO) AS cdas_panel,
       COUNT(DISTINCT ID_DEBITO) FILTER (WHERE paid > 0) AS cdas_paid_panel,
       SUM(paid) AS paid_panel, SUM(stock) AS stock_panel
FROM pf GROUP BY ALL''')
st_raw["n_min"] = st_raw[["cnpjs_dec2025", "cnpjs_panel"]].replace(0, np.nan).min(axis=1).fillna(0)
st_s, n_sup_status = suppress(st_raw, "status", "n_min")
# re-aggregating sums is exact for CDAs, stock and payments; distinct CNPJs of pooled rows are summed (upper bound)
st_tab = st_s.groupby(["origin", "status"], as_index=False)[
    ["cnpjs_dec2025", "cdas_dec2025", "stock_dec2025", "cnpjs_panel", "cdas_panel", "cdas_paid_panel", "paid_panel", "stock_panel"]].sum()
st_tab = blank_small_pool(st_tab, "status", ["cnpjs_dec2025", "cnpjs_panel"])
st_tab["pct_origin_stock_dec2025"] = 100 * st_tab.stock_dec2025 / st_tab.groupby("origin").stock_dec2025.transform("sum")
st_tab["pooled_recovery_pct"] = 100 * st_tab.paid_panel / st_tab.stock_panel
st_tab["pct_cdas_any_payment"] = 100 * st_tab.cdas_paid_panel / st_tab.cdas_panel
st_tab["stock_dec2025_bn"] = st_tab.stock_dec2025 / 1e9
order = {s: i for i, s in enumerate(STATUS_ORDER + ["other (suppressed)"])}
st_tab = st_tab.sort_values(["origin", "status"], key=lambda c: c.map(order) if c.name == "status" else c).reset_index(drop=True)
st_out = st_tab[["origin", "status", "cnpjs_dec2025", "cdas_dec2025", "stock_dec2025_bn", "pct_origin_stock_dec2025",
                 "cdas_panel", "pct_cdas_any_payment", "pooled_recovery_pct"]]
save_tab(st_out, "tab_status_recovery", fmts={"cnpjs_dec2025": 0, "cdas_dec2025": 0, "stock_dec2025_bn": 2,
                                              "pct_origin_stock_dec2025": 1, "cdas_panel": 0, "pct_cdas_any_payment": 1,
                                              "pooled_recovery_pct": 2},
         caption="Registration status of the debtor CNPJ (RFB, March 2026) by origin: Dec 2025 stock (BRL bn) and 2016-2025 recovery")
sr = st_tab.set_index(["origin", "status"])
for o, k in (("assessed", "Assessed"), ("declared", "Declared")):
    for s_, sk in (("active", "Active"), ("unfit (inapta)", "Unfit"), ("closed (baixada)", "Closed"), ("suspended", "Suspended")):
        if (o, s_) in sr.index:
            reg(f"stPctStock{k}{sk}", sr.loc[(o, s_), "pct_origin_stock_dec2025"], 1, "tab_status_recovery")
            reg(f"stRecovery{k}{sk}", sr.loc[(o, s_), "pooled_recovery_pct"], 2, "tab_status_recovery")
            reg(f"stAnyPay{k}{sk}", sr.loc[(o, s_), "pct_cdas_any_payment"], 1, "tab_status_recovery")
            reg(f"stCnpj{k}{sk}", sr.loc[(o, s_), "cnpjs_dec2025"], 0, "tab_status_recovery")
reg("stSuppressedCells", n_sup_status, 0, "tab_status_recovery")
st_out

In [ ]:
CENSOR = "2025-12-31"
km_raw = q(f'''
WITH fp AS (SELECT ID_DEBITO, MIN(first_pay) AS first_pay FROM gare GROUP BY 1),
c AS (
  SELECT c.origin, c.status, c.cnpj,
         CASE WHEN fp.first_pay IS NOT NULL THEN GREATEST(DATE_DIFF('month', c.dt_insc, fp.first_pay), 0)
              ELSE DATE_DIFF('month', c.dt_insc, DATE '{CENSOR}') END AS t,
         (fp.first_pay IS NOT NULL)::INTEGER AS event
  FROM cda_firm c LEFT JOIN fp USING (ID_DEBITO)
  WHERE c.origin <> 'other' AND c.dt_insc BETWEEN DATE '{YEAR_MIN}-01-01' AND DATE '{CENSOR}')
SELECT origin, status, t, COUNT(*) AS n, SUM(event) AS d FROM c GROUP BY ALL''')
km_cnpj = q(f'''
SELECT origin, status, COUNT(DISTINCT cnpj) AS cnpjs FROM cda_firm
WHERE origin <> 'other' AND dt_insc BETWEEN DATE '{YEAR_MIN}-01-01' AND DATE '{CENSOR}' GROUP BY ALL''')
km_raw = km_raw.merge(km_cnpj, on=["origin", "status"])
km_raw.loc[km_raw.cnpjs < 5, "status"] = "other (suppressed)"
pool_n = km_cnpj[km_cnpj.cnpjs < 5].groupby("origin").cnpjs.sum()
km_raw = km_raw[~((km_raw.status == "other (suppressed)") & km_raw.origin.map(pool_n).fillna(0).lt(5))]
km_raw = km_raw.groupby(["origin", "status", "t"], as_index=False)[["n", "d"]].sum()

def kaplan_meier(g):
    g = g.sort_values("t")
    at_risk = g.n[::-1].cumsum()[::-1].to_numpy()          # CDAs with T >= t
    surv = np.cumprod(1 - g.d.to_numpy() / at_risk)
    return pd.DataFrame({"t": g.t.to_numpy(), "paid_share": 1 - surv, "at_risk": at_risk})

curves, rows = [], []
for (o, s_), g in km_raw.groupby(["origin", "status"]):
    c = kaplan_meier(g)
    c.insert(0, "status", s_); c.insert(0, "origin", o)
    curves.append(c)
    at = lambda m: c.paid_share[c.t <= m].max() if (c.t <= m).any() else 0.0
    med = c.t[c.paid_share >= 0.5].min()
    rows.append({"origin": o, "status": s_, "cdas_cohort": int(g.n.sum()), "first_payments": int(g.d.sum()),
                 "pct_paid_12m": 100 * at(12), "pct_paid_24m": 100 * at(24), "pct_paid_60m": 100 * at(60),
                 "median_months": med if pd.notna(med) else np.nan})
km_curves = pd.concat(curves, ignore_index=True)
km_tab = pd.DataFrame(rows)
km_tab = km_tab.sort_values(["origin", "status"], key=lambda c: c.map(order) if c.name == "status" else c).reset_index(drop=True)
save_tab(km_tab, "tab_km_status_origin", fmts={"cdas_cohort": 0, "first_payments": 0, "pct_paid_12m": 1, "pct_paid_24m": 1,
                                               "pct_paid_60m": 1, "median_months": 0},
         caption="Kaplan-Meier time to first payment, enrolment cohort 2016-2025, by origin and registration status (RFB, March 2026); empty median = not reached")
kt = km_tab.set_index(["origin", "status"])
for o, k in (("assessed", "Assessed"), ("declared", "Declared")):
    for s_, sk in (("active", "Active"), ("unfit (inapta)", "Unfit"), ("closed (baixada)", "Closed")):
        if (o, s_) in kt.index:
            reg(f"kmPaidOneYr{k}{sk}", kt.loc[(o, s_), "pct_paid_12m"], 1, "tab_km_status_origin")
            reg(f"kmPaidFiveYr{k}{sk}", kt.loc[(o, s_), "pct_paid_60m"], 1, "tab_km_status_origin")
            reg(f"kmCohort{k}{sk}", kt.loc[(o, s_), "cdas_cohort"], 0, "tab_km_status_origin")
    reg(f"kmCohortN{k}", km_tab[km_tab.origin == o].cdas_cohort.sum(), 0, "tab_km_status_origin")
km_tab

**How to read Figure `fig_km_status_origin`.** Two panels, assessed (a) and declared (b). Each line is a registration status; the vertical axis is the Kaplan–Meier share of CDAs of the 2016–2025 enrolment cohort that have made a first payment by month $t$ after enrolment (0–120 months). Statuses with fewer than 100 CDAs in the cohort are not drawn (they stay in the table), and each curve stops when fewer than 20 CDAs remain at risk. **What not to conclude:** the status is observed in March 2026, after the outcome, so a lower curve for closed or unfit firms is an association, not the effect of being closed; curves end where the cohort runs out of follow-up, and the right tail rests on the oldest cohorts only. **[ÉTORE DECIDE]** whether the article reports this as evidence that assessed debt of active firms is recovered more slowly than declared debt of the same status.

In [ ]:
ST_COL = {"active": CEMEPI["deep_teal"], "unfit (inapta)": CEMEPI["sand"], "suspended": CEMEPI["green"],
          "closed (baixada)": CEMEPI["mid_teal"], "null (nula)": "#B5651D", "other code": CEMEPI["grey"],
          "root only (status unknown)": CEMEPI["grey"], "not found in register": "#4D4D4D", "other (suppressed)": "#B0B0B0"}
big = km_tab[km_tab.cdas_cohort >= 100][["origin", "status"]]
plot_km = km_curves.merge(big, on=["origin", "status"]).query("t <= 120 and at_risk >= 20")
fig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.06,
                    subplot_titles=("(a) Assessed", "(b) Declared"))
shown_leg = set()
for cc, o in ((1, "assessed"), (2, "declared")):
    for s_ in STATUS_ORDER + ["other (suppressed)"]:
        d = plot_km[(plot_km.origin == o) & (plot_km.status == s_)]
        if len(d):
            fig.add_scatter(x=d.t, y=100 * d.paid_share, mode="lines", line_shape="hv", name=s_, legendgroup=s_,
                            showlegend=s_ not in shown_leg, line=dict(color=ST_COL[s_], dash="dot" if "register" in s_ or "root" in s_ else "solid"),
                            row=1, col=cc)
            shown_leg.add(s_)
fig.update_xaxes(title_text="months since enrolment", dtick=12, range=[0, 120])
fig.update_yaxes(title_text="% with a first payment", range=[0, 100], row=1, col=1)
save_fig(fig, "fig_km_status_origin", plot_km.assign(paid_pct=lambda d: 100 * d.paid_share)[["origin", "status", "t", "paid_pct", "at_risk"]],
         width=760, height=380)

### 10.2 Economic activity (CNAE division) and origin

**Method.** CNAE 2.0 main activity of the debtor establishment, at **division** level (2 digits; IBGE/CONCLA), crossed with origin. **Division names.** IBGE publishes the CNAE 2.0 structure only in Portuguese (no official English table of the divisions was found on the IBGE or CONCLA pages on 27 Oct 2026); the English names in the tables are a literal translation of the official Portuguese names of the 87 divisions and 21 sections, retrieved from the IBGE API (`servicodados.ibge.gov.br/api/v2/cnae/divisoes` and `/secoes`), kept alongside in column `division_name_pt`. For each division: distinct CNPJs, December 2025 stock by origin, the **assessed weight** $w_d = 100\,E^{\text{assessed}}_d/(E^{\text{assessed}}_d+E^{\text{declared}}_d)$ (share of the division's stock that comes from infringement notices), and the pooled recovery rate $\bar\rho$ (2016–2025 panel) by origin and in total. CDAs without an establishment-level match have no CNAE and form the row `no CNAE (not matched)`. Divisions whose assessed cell is empty have an assessed weight of 0 and no assessed recovery rate.

**Disclosure control.** A division is pooled into `other (suppressed)` if any of its origin cells has 1–4 distinct CNPJs, in the December 2025 snapshot or in the panel; the whole division is pooled (not only the small cell) because the assessed weight of a division with one suppressed cell would reveal it. A pooled row that still has fewer than 5 CNPJs is blanked.

**Across divisions.** The Spearman rank correlation between $w_d$ and the division's total pooled recovery rate, over the divisions shown, describes whether divisions whose stock is mostly assessed also show a lower pooled recovery rate; it is a description of the cross-section, not an effect of activity or of origin (divisions are not independent draws and are not weighted).

In [ ]:
# CNAE 2.0 (IBGE/CONCLA): literal English translation of the official Portuguese names (IBGE API /api/v2/cnae/divisoes and /secoes,
# retrieved 27 Oct 2026); IBGE publishes no official English table of the divisions. Portuguese name kept for traceability.
SECTION_EN = {'A': 'Agriculture, livestock, forestry production, fishing and aquaculture', 'B': 'Extractive industries', 'C': 'Manufacturing industries', 'D': 'Electricity and gas', 'E': 'Water, sewerage, waste management and remediation activities', 'F': 'Construction', 'G': 'Trade; repair of motor vehicles and motorcycles', 'H': 'Transport, storage and mail', 'I': 'Accommodation and food services', 'J': 'Information and communication', 'K': 'Financial, insurance and related service activities', 'L': 'Real estate activities', 'M': 'Professional, scientific and technical activities', 'N': 'Administrative activities and complementary services', 'O': 'Public administration, defence and social security', 'P': 'Education', 'Q': 'Human health and social services', 'R': 'Arts, culture, sport and recreation', 'S': 'Other service activities', 'T': 'Domestic services', 'U': 'International organisations and other extraterritorial institutions'}
DIVISION_EN = {'01': 'Agriculture, livestock and related services', '02': 'Forestry production', '03': 'Fishing and aquaculture', '05': 'Mining of coal', '06': 'Extraction of oil and natural gas', '07': 'Mining of metal ores', '08': 'Mining of non-metallic minerals', '09': 'Support activities for mineral extraction', '10': 'Manufacture of food products', '11': 'Manufacture of beverages', '12': 'Manufacture of tobacco products', '13': 'Manufacture of textile products', '14': 'Manufacture of wearing apparel and accessories', '15': 'Preparation of leather and manufacture of leather goods, travel goods and footwear', '16': 'Manufacture of wood products', '17': 'Manufacture of pulp, paper and paper products', '18': 'Printing and reproduction of recordings', '19': 'Manufacture of coke, petroleum products and biofuels', '20': 'Manufacture of chemical products', '21': 'Manufacture of pharmochemical and pharmaceutical products', '22': 'Manufacture of rubber and plastic products', '23': 'Manufacture of non-metallic mineral products', '24': 'Metallurgy', '25': 'Manufacture of metal products, except machinery and equipment', '26': 'Manufacture of computer equipment, electronic and optical products', '27': 'Manufacture of electrical machinery, apparatus and materials', '28': 'Manufacture of machinery and equipment', '29': 'Manufacture of motor vehicles, trailers and bodies', '30': 'Manufacture of other transport equipment, except motor vehicles', '31': 'Manufacture of furniture', '32': 'Manufacture of miscellaneous products', '33': 'Maintenance, repair and installation of machinery and equipment', '35': 'Electricity, gas and other utilities', '36': 'Water collection, treatment and distribution', '37': 'Sewerage and related activities', '38': 'Waste collection, treatment and disposal; materials recovery', '39': 'Remediation and other waste management services', '41': 'Construction of buildings', '42': 'Infrastructure works', '43': 'Specialised construction services', '45': 'Trade and repair of motor vehicles and motorcycles', '46': 'Wholesale trade, except motor vehicles and motorcycles', '47': 'Retail trade', '49': 'Land transport', '50': 'Water transport', '51': 'Air transport', '52': 'Storage and auxiliary transport activities', '53': 'Mail and other delivery activities', '55': 'Accommodation', '56': 'Food services', '58': 'Publishing and publishing integrated with printing', '59': 'Motion picture, video and television programme production; sound recording and music publishing', '60': 'Radio and television activities', '61': 'Telecommunications', '62': 'Information technology service activities', '63': 'Information service activities', '64': 'Financial service activities', '65': 'Insurance, reinsurance, supplementary pension and health plans', '66': 'Activities auxiliary to financial services, insurance, supplementary pension and health plans', '68': 'Real estate activities', '69': 'Legal, accounting and auditing activities', '70': 'Activities of head offices and business management consultancy', '71': 'Architecture and engineering services; technical testing and analysis', '72': 'Scientific research and development', '73': 'Advertising and market research', '74': 'Other professional, scientific and technical activities', '75': 'Veterinary activities', '77': 'Non-real-estate rental and management of non-financial intangible assets', '78': 'Selection, placement and leasing of labour', '79': 'Travel agencies, tour operators and reservation services', '80': 'Surveillance, security and investigation activities', '81': 'Services to buildings and landscaping activities', '82': 'Office and administrative support services and other services mainly to businesses', '84': 'Public administration, defence and social security', '85': 'Education', '86': 'Human health care activities', '87': 'Human health care activities integrated with social assistance, provided in collective and private residences', '88': 'Social assistance services without accommodation', '90': 'Artistic, creative and entertainment activities', '91': 'Activities related to cultural and environmental heritage', '92': 'Gambling and betting activities', '93': 'Sports, recreation and leisure activities', '94': 'Activities of membership organisations', '95': 'Repair and maintenance of computer and communication equipment and of personal and household goods', '96': 'Other personal service activities', '97': 'Domestic services', '99': 'International organisations and other extraterritorial institutions'}
DIVISION_PT = {'01': 'AGRICULTURA, PECUÁRIA E SERVIÇOS RELACIONADOS', '02': 'PRODUÇÃO FLORESTAL', '03': 'PESCA E AQÜICULTURA', '05': 'EXTRAÇÃO DE CARVÃO MINERAL', '06': 'EXTRAÇÃO DE PETRÓLEO E GÁS NATURAL', '07': 'EXTRAÇÃO DE MINERAIS METÁLICOS', '08': 'EXTRAÇÃO DE MINERAIS NÃO-METÁLICOS', '09': 'ATIVIDADES DE APOIO À EXTRAÇÃO DE MINERAIS', '10': 'FABRICAÇÃO DE PRODUTOS ALIMENTÍCIOS', '11': 'FABRICAÇÃO DE BEBIDAS', '12': 'FABRICAÇÃO DE PRODUTOS DO FUMO', '13': 'FABRICAÇÃO DE PRODUTOS TÊXTEIS', '14': 'CONFECÇÃO DE ARTIGOS DO VESTUÁRIO E ACESSÓRIOS', '15': 'PREPARAÇÃO DE COUROS E FABRICAÇÃO DE ARTEFATOS DE COURO, ARTIGOS PARA VIAGEM E CALÇADOS', '16': 'FABRICAÇÃO DE PRODUTOS DE MADEIRA', '17': 'FABRICAÇÃO DE CELULOSE, PAPEL E PRODUTOS DE PAPEL', '18': 'IMPRESSÃO E REPRODUÇÃO DE GRAVAÇÕES', '19': 'FABRICAÇÃO DE COQUE, DE PRODUTOS DERIVADOS DO PETRÓLEO E DE BIOCOMBUSTÍVEIS', '20': 'FABRICAÇÃO DE PRODUTOS QUÍMICOS', '21': 'FABRICAÇÃO DE PRODUTOS FARMOQUÍMICOS E FARMACÊUTICOS', '22': 'FABRICAÇÃO DE PRODUTOS DE BORRACHA E DE MATERIAL PLÁSTICO', '23': 'FABRICAÇÃO DE PRODUTOS DE MINERAIS NÃO-METÁLICOS', '24': 'METALURGIA', '25': 'FABRICAÇÃO DE PRODUTOS DE METAL, EXCETO MÁQUINAS E EQUIPAMENTOS', '26': 'FABRICAÇÃO DE EQUIPAMENTOS DE INFORMÁTICA, PRODUTOS ELETRÔNICOS E ÓPTICOS', '27': 'FABRICAÇÃO DE MÁQUINAS, APARELHOS E MATERIAIS ELÉTRICOS', '28': 'FABRICAÇÃO DE MÁQUINAS E EQUIPAMENTOS', '29': 'FABRICAÇÃO DE VEÍCULOS AUTOMOTORES, REBOQUES E CARROCERIAS', '30': 'FABRICAÇÃO DE OUTROS EQUIPAMENTOS DE TRANSPORTE, EXCETO VEÍCULOS AUTOMOTORES', '31': 'FABRICAÇÃO DE MÓVEIS', '32': 'FABRICAÇÃO DE PRODUTOS DIVERSOS', '33': 'MANUTENÇÃO, REPARAÇÃO E INSTALAÇÃO DE MÁQUINAS E EQUIPAMENTOS', '35': 'ELETRICIDADE, GÁS E OUTRAS UTILIDADES', '36': 'CAPTAÇÃO, TRATAMENTO E DISTRIBUIÇÃO DE ÁGUA', '37': 'ESGOTO E ATIVIDADES RELACIONADAS', '38': 'COLETA, TRATAMENTO E DISPOSIÇÃO DE RESÍDUOS; RECUPERAÇÃO DE MATERIAIS', '39': 'DESCONTAMINAÇÃO E OUTROS SERVIÇOS DE GESTÃO DE RESÍDUOS', '41': 'CONSTRUÇÃO DE EDIFÍCIOS', '42': 'OBRAS DE INFRA-ESTRUTURA', '43': 'SERVIÇOS ESPECIALIZADOS PARA CONSTRUÇÃO', '45': 'COMÉRCIO E REPARAÇÃO DE VEÍCULOS AUTOMOTORES E MOTOCICLETAS', '46': 'COMÉRCIO POR ATACADO, EXCETO VEÍCULOS AUTOMOTORES E MOTOCICLETAS', '47': 'COMÉRCIO VAREJISTA', '49': 'TRANSPORTE TERRESTRE', '50': 'TRANSPORTE AQUAVIÁRIO', '51': 'TRANSPORTE AÉREO', '52': 'ARMAZENAMENTO E ATIVIDADES AUXILIARES DOS TRANSPORTES', '53': 'CORREIO E OUTRAS ATIVIDADES DE ENTREGA', '55': 'ALOJAMENTO', '56': 'ALIMENTAÇÃO', '58': 'EDIÇÃO E EDIÇÃO INTEGRADA À IMPRESSÃO', '59': 'ATIVIDADES CINEMATOGRÁFICAS, PRODUÇÃO DE VÍDEOS E DE PROGRAMAS DE TELEVISÃO; GRAVAÇÃO DE SOM E EDIÇÃO DE MÚSICA', '60': 'ATIVIDADES DE RÁDIO E DE TELEVISÃO', '61': 'TELECOMUNICAÇÕES', '62': 'ATIVIDADES DOS SERVIÇOS DE TECNOLOGIA DA INFORMAÇÃO', '63': 'ATIVIDADES DE PRESTAÇÃO DE SERVIÇOS DE INFORMAÇÃO', '64': 'ATIVIDADES DE SERVIÇOS FINANCEIROS', '65': 'SEGUROS, RESSEGUROS, PREVIDÊNCIA COMPLEMENTAR E PLANOS DE SAÚDE', '66': 'ATIVIDADES AUXILIARES DOS SERVIÇOS FINANCEIROS, SEGUROS, PREVIDÊNCIA COMPLEMENTAR E PLANOS DE SAÚDE', '68': 'ATIVIDADES IMOBILIÁRIAS', '69': 'ATIVIDADES JURÍDICAS, DE CONTABILIDADE E DE AUDITORIA', '70': 'ATIVIDADES DE SEDES DE EMPRESAS E DE CONSULTORIA EM GESTÃO EMPRESARIAL', '71': 'SERVIÇOS DE ARQUITETURA E ENGENHARIA; TESTES E ANÁLISES TÉCNICAS', '72': 'PESQUISA E DESENVOLVIMENTO CIENTÍFICO', '73': 'PUBLICIDADE E PESQUISA DE MERCADO', '74': 'OUTRAS ATIVIDADES PROFISSIONAIS, CIENTÍFICAS E TÉCNICAS', '75': 'ATIVIDADES VETERINÁRIAS', '77': 'ALUGUÉIS NÃO-IMOBILIÁRIOS E GESTÃO DE ATIVOS INTANGÍVEIS NÃO-FINANCEIROS', '78': 'SELEÇÃO, AGENCIAMENTO E LOCAÇÃO DE MÃO-DE-OBRA', '79': 'AGÊNCIAS DE VIAGENS, OPERADORES TURÍSTICOS E SERVIÇOS DE RESERVAS', '80': 'ATIVIDADES DE VIGILÂNCIA, SEGURANÇA E INVESTIGAÇÃO', '81': 'SERVIÇOS PARA EDIFÍCIOS E ATIVIDADES PAISAGÍSTICAS', '82': 'SERVIÇOS DE ESCRITÓRIO, DE APOIO ADMINISTRATIVO E OUTROS SERVIÇOS PRESTADOS PRINCIPALMENTE ÀS EMPRESAS', '84': 'ADMINISTRAÇÃO PÚBLICA, DEFESA E SEGURIDADE SOCIAL', '85': 'EDUCAÇÃO', '86': 'ATIVIDADES DE ATENÇÃO À SAÚDE HUMANA', '87': 'ATIVIDADES DE ATENÇÃO À SAÚDE HUMANA INTEGRADAS COM ASSISTÊNCIA SOCIAL, PRESTADAS EM RESIDÊNCIAS COLETIVAS E PARTICULARES', '88': 'SERVIÇOS DE ASSISTÊNCIA SOCIAL SEM ALOJAMENTO', '90': 'ATIVIDADES ARTÍSTICAS, CRIATIVAS E DE ESPETÁCULOS', '91': 'ATIVIDADES LIGADAS AO PATRIMÔNIO CULTURAL E AMBIENTAL', '92': 'ATIVIDADES DE EXPLORAÇÃO DE JOGOS DE AZAR E APOSTAS', '93': 'ATIVIDADES ESPORTIVAS E DE RECREAÇÃO E LAZER', '94': 'ATIVIDADES DE ORGANIZAÇÕES ASSOCIATIVAS', '95': 'REPARAÇÃO E MANUTENÇÃO DE EQUIPAMENTOS DE INFORMÁTICA E COMUNICAÇÃO E DE OBJETOS PESSOAIS E DOMÉSTICOS', '96': 'OUTRAS ATIVIDADES DE SERVIÇOS PESSOAIS', '97': 'SERVIÇOS DOMÉSTICOS', '99': 'ORGANISMOS INTERNACIONAIS E OUTRAS INSTITUIÇÕES EXTRATERRITORIAIS'}
dv = q('''
SELECT COALESCE(division, 'none') AS division, ANY_VALUE(section) AS section, origin,
       COUNT(DISTINCT cnpj) FILTER (WHERE in_dec2025) AS cnpjs_dec2025,
       COALESCE(SUM(stock) FILTER (WHERE in_dec2025), 0) AS stock_dec2025,
       COUNT(DISTINCT cnpj) AS cnpjs_panel, SUM(paid) AS paid_panel, SUM(stock) AS stock_panel
FROM pf GROUP BY ALL''')
dv_cnpj = q('''
SELECT COALESCE(division, 'none') AS division, COUNT(DISTINCT cnpj) FILTER (WHERE in_dec2025) AS cnpjs_dec2025_all,
       COUNT(DISTINCT cnpj) AS cnpjs_panel_all
FROM pf GROUP BY ALL''')
small = dv[((dv.cnpjs_dec2025 > 0) & (dv.cnpjs_dec2025 < 5)) | ((dv.cnpjs_panel > 0) & (dv.cnpjs_panel < 5))].division.unique()
small = [d for d in small if d != "none"]
dv["label"] = np.where(dv.division == "none", "no CNAE (not matched)",
                       np.where(dv.division.isin(small), "other (suppressed)",
                                "div " + dv.division + " (" + dv.section.fillna("?") + ")"))
dv_cnpj["label"] = dv_cnpj.division.map(dv.drop_duplicates("division").set_index("division").label)
w = dv.pivot_table(index="label", columns="origin", values=["stock_dec2025", "paid_panel", "stock_panel"], aggfunc="sum", fill_value=0)
w.columns = [f"{a}_{b}" for a, b in w.columns]
w = w.join(dv_cnpj.groupby("label")[["cnpjs_dec2025_all", "cnpjs_panel_all"]].sum())
for o in ("assessed", "declared"):
    for c in ("stock_dec2025", "paid_panel", "stock_panel"):
        if f"{c}_{o}" not in w:
            w[f"{c}_{o}"] = 0.0
tab_dv = pd.DataFrame({
    "division": w.index,
    "division_name": [DIVISION_EN[l.split()[1]] if l.startswith("div ") else "" for l in w.index],
    "division_name_pt": [DIVISION_PT[l.split()[1]] if l.startswith("div ") else "" for l in w.index],
    "section_name": [SECTION_EN.get(l.split("(")[-1].rstrip(")"), "") if l.startswith("div ") else "" for l in w.index],
    "cnpjs_dec2025": w.cnpjs_dec2025_all.to_numpy(), "cnpjs_panel": w.cnpjs_panel_all.to_numpy(),
    "stock_dec2025_bn": (w.stock_dec2025_assessed + w.stock_dec2025_declared).to_numpy() / 1e9,
    "assessed_weight_pct": (100 * w.stock_dec2025_assessed / (w.stock_dec2025_assessed + w.stock_dec2025_declared)).to_numpy(),
    "recovery_assessed_pct": (100 * w.paid_panel_assessed / w.stock_panel_assessed.replace(0, np.nan)).to_numpy(),
    "recovery_declared_pct": (100 * w.paid_panel_declared / w.stock_panel_declared.replace(0, np.nan)).to_numpy(),
    "recovery_all_pct": (100 * (w.paid_panel_assessed + w.paid_panel_declared)
                         / (w.stock_panel_assessed + w.stock_panel_declared)).to_numpy()})
tab_dv = blank_small_pool(tab_dv, "division", ["cnpjs_dec2025", "cnpjs_panel"])
tab_dv["pct_vat_stock_dec2025"] = 100 * tab_dv.stock_dec2025_bn / tab_dv.stock_dec2025_bn.sum()
special = tab_dv.division.isin(["other (suppressed)", "no CNAE (not matched)"])
tab_dv = pd.concat([tab_dv[~special].sort_values("stock_dec2025_bn", ascending=False), tab_dv[special]]).reset_index(drop=True)
save_tab(tab_dv, "tab_cnae_division_origin",
         fmts={"cnpjs_dec2025": 0, "cnpjs_panel": 0, "stock_dec2025_bn": 2, "assessed_weight_pct": 1,
               "recovery_assessed_pct": 2, "recovery_declared_pct": 2, "recovery_all_pct": 2, "pct_vat_stock_dec2025": 1},
         caption="CNAE division (RFB, March 2026) x origin: Dec 2025 stock (BRL bn), assessed weight and pooled 2016-2025 recovery; divisions with a cell of 1-4 CNPJs pooled")
shown = tab_dv[~special & tab_dv.recovery_all_pct.notna()]
rho, p_rho = stats.spearmanr(shown.assessed_weight_pct, shown.recovery_all_pct)
reg("cnaeNDivShown", len(shown), 0, "tab_cnae_division_origin")
reg("cnaeNDivSuppressed", len(small), 0, "tab_cnae_division_origin")
reg("cnaeSpearman", rho, 2, "tab_cnae_division_origin")
reg("cnaePctStockNoCnae", tab_dv.loc[tab_dv.division == "no CNAE (not matched)", "pct_vat_stock_dec2025"].sum(), 1, "tab_cnae_division_origin")
top5 = shown.head(5)
reg("cnaeTopFivePctStock", top5.pct_vat_stock_dec2025.sum(), 1, "tab_cnae_division_origin")
reg("cnaeTopDivision", top5.division.iloc[0].split()[1], source="tab_cnae_division_origin")
reg("cnaeTopDivisionName", top5.division_name.iloc[0].lower(), source="tab_cnae_division_origin (English name, IBGE CNAE 2.0)")
reg("cnaeSecondDivision", top5.division.iloc[1].split()[1], source="tab_cnae_division_origin")
reg("cnaeSecondDivisionName", top5.division_name.iloc[1].lower(), source="tab_cnae_division_origin (English name, IBGE CNAE 2.0)")
reg("cnaeSecondDivisionPctStock", top5.pct_vat_stock_dec2025.iloc[1], 1, "tab_cnae_division_origin")
reg("cnaeSecondDivisionAssessedWeight", top5.assessed_weight_pct.iloc[1], 1, "tab_cnae_division_origin")
reg("cnaeTopDivisionPctStock", top5.pct_vat_stock_dec2025.iloc[0], 1, "tab_cnae_division_origin")
reg("cnaeTopDivisionAssessedWeight", top5.assessed_weight_pct.iloc[0], 1, "tab_cnae_division_origin")
reg("cnaeAssessedWeightMin", shown.assessed_weight_pct.min(), 1, "tab_cnae_division_origin")
reg("cnaeAssessedWeightMax", shown.assessed_weight_pct.max(), 1, "tab_cnae_division_origin")
print(f"divisions shown {len(shown)}, pooled {len(small)}; Spearman rho = {rho:.2f} (p = {p_rho:.3g})")
tab_dv

**How to read Figure `fig_cnae_division_origin`.** One bubble per CNAE division shown in the table (pooled and unmatched rows excluded). Horizontal axis: assessed weight $w_d$ (% of the division's December 2025 VAT (ICMS) stock that is assessed). Vertical axis: pooled 2016–2025 recovery rate of the division, all origins, log scale. Bubble area is proportional to the division's December 2025 stock; the ten largest divisions are labelled with their code. **What not to conclude:** a division's position mixes its origin mix with its firms' size, age and status; the cross-division correlation is ecological and does not say that assessed debt recovers less *within* a division (the table gives the within-division rates by origin). **[ÉTORE DECIDE]** which divisions, if any, the article names.

In [ ]:
d = shown.copy()
d["label"] = np.where(d.index < 10, d.division.str.extract(r"div (\d+)")[0], "")
fig = go.Figure(go.Scatter(x=d.assessed_weight_pct, y=d.recovery_all_pct, mode="markers+text", text=d.label,
                           textposition="top center", textfont=dict(size=9),
                           marker=dict(size=d.stock_dec2025_bn, sizemode="area", sizeref=2 * d.stock_dec2025_bn.max() / 40 ** 2,
                                       sizemin=3, color=CEMEPI["teal"], opacity=0.7, line=dict(width=0.5, color=CEMEPI["deep_teal"]))))
fig.update_xaxes(title_text="assessed share of Dec 2025 stock (%)", range=[-2, 102])
fig.update_yaxes(title_text="pooled recovery 2016-2025 (%, log)", type="log",
                 tickvals=[0.01, 0.03, 0.1, 0.3, 1, 3], ticktext=["0.01", "0.03", "0.1", "0.3", "1", "3"])
fig.update_layout(showlegend=False)
save_fig(fig, "fig_cnae_division_origin", d[["division", "division_name", "section_name", "cnpjs_dec2025", "stock_dec2025_bn",
                                             "assessed_weight_pct", "recovery_all_pct"]], height=380)

## 11. Comparison with the 27 Oct 2026 draft

The draft's numbers came from frozen lab outputs (commit `a6b4b8a`) and are used **only as a check**: nothing from the lab is read here; the values below were typed from the draft. Column `rel_diff_pct` is $100\,(\text{new}-\text{draft})/|\text{draft}|$, computed on the unrounded new value (column `new_in_numeros` shows the rounded text that goes to `numeros.tex`); differences up to 0.5% are treated as rounding of the draft.

In [ ]:
def num(key):
    return RAW[key]
DRAFT = [  # (key in numeros.tex, value in the draft, where it came from in the draft)
    ("stockVatBn", 429.943, "CONC ICMS/CDA stock"), ("nCdaVat", 2323367, "CONC ICMS/CDA units"),
    ("topOneCdaVat", 75.863, "CONC ICMS/CDA top 1%"), ("kFiftyDebtorVat", 550, "CONC ICMS/Debtor k50"),
    ("stockAssessedBn", 313.303, "CONC Autuação stock"), ("nCdaAssessed", 59536, "CONC Autuação units"),
    ("stockDeclaredBn", 116.436, "CONC Declarado stock"), ("nCdaDeclared", 2254464, "CONC Declarado units"),
    ("kFiftyDebtorAssessed", 302, "CONC Autuação/Debtor k50"), ("kFiftyDebtorDeclared", 982, "CONC Declarado/Debtor k50"),
    ("topOneDebtorAssessed", 49.497, "CONC Autuação/Debtor top 1%"), ("topOneDebtorDeclared", 58.900, "CONC Declarado/Debtor top 1%"),
    ("pctCdaAssessed", 2.5716, "A72 pct_cda"), ("pctStockAssessed", 72.8709, "A72 pct_stock"),
    ("pctCdaPj", 99.949, "01_dados c25 pct_cda"), ("pctStockPj", 99.520, "01_dados c25 pct_stock"),
    ("meanValueAssessed", 5300275, "tab_icms_raw_ticket"), ("medianValueAssessed", 589090, "tab_icms_raw_ticket"),
    ("meanValueDeclared", 51579, "tab_icms_raw_ticket"), ("medianValueDeclared", 7308, "tab_icms_raw_ticket"),
    ("medianAgeAssessed", 9.31, "tab_icms_raw_age"), ("medianAgeDeclared", 4.61, "tab_icms_raw_age"),
    ("pctAnyPaymentAssessed", 14.438, "tab_icms_raw_payment"), ("pctAnyPaymentDeclared", 25.885, "tab_icms_raw_payment"),
    ("yearsFirstPaymentAssessed", 3.337, "tab_icms_raw_payment"), ("yearsFirstPaymentDeclared", 0.186, "tab_icms_raw_payment"),
    ("pctCdaFiledAssessed", 88.43, "tab_icms_raw_filing"), ("pctCdaFiledDeclared", 69.72, "tab_icms_raw_filing"),
    ("pctStockFiledAssessed", 91.46, "tab_icms_raw_filing"), ("pctStockFiledDeclared", 94.19, "tab_icms_raw_filing"),
    ("rBisValue", 0.887, "tab_icms_grouping_decision"), ("rBisAge", 0.399, "tab_icms_grouping_decision"),
    ("rBisRate", 0.126, "tab_icms_grouping_decision"), ("kwRateH", 14600, "tab_icms_raw_kruskal rate H=1.46e+04"),
    ("rateMinDeclared", 0.768, "SER declared min"), ("rateMaxDeclared", 1.586, "SER declared max"),
    ("rateMinAssessed", 0.033, "SER assessed min"), ("rateMaxAssessed", 0.390, "SER assessed max"),
    ("rateTwentyFiveDeclared", 1.5144, "SER 2025"), ("rateTwentyFiveAssessed", 0.2139, "SER 2025"),
    ("cdaAssessedNineteen", 124498, "SER assessed n_cda 2019"), ("cdaAssessedTwenty", 67224, "SER assessed n_cda 2020"),
    ("paidAssessedNineteen", 0.701, "SER assessed paid 2019"), ("paidAssessedTwentyFour", 1.184, "SER assessed paid 2024"),
    ("topOnePctCdaAssessed", 78.68, "tab_top_composition (type Autuação)"), ("topOnePctValueAssessed", 90.66, "tab_top_composition (type Autuação)"),
    ("topTenthPctCdaAssessed", 93.895, "B43 top 0.1%"), ("restOnePctCdaAssessed", 1.792, "B43 remainder (outside top 1%)"),
    ("kmN", 23253, "KM base"), ("kmMinValueMn", 1.58, "KM base"), ("kmSumBn", 324.63, "KM base"), ("kmPctPaid", 12.46, "KM base"),
    ("kmSilEuclid", 0.6814, "KM k=4"), ("kmSilGower", 0.6645, "KM k=4"),
    ("kmGOnePctCda", 70.6, "KM G1"), ("kmGOnePctValue", 79.7, "KM G1"), ("kmGTwoPctCda", 19.7, "KM G2"), ("kmGTwoPctValue", 8.8, "KM G2"),
    ("kmGThreePctCda", 6.4, "KM G3"), ("kmGThreePctValue", 6.1, "KM G3"), ("kmGFourPctCda", 3.3, "KM G4"), ("kmGFourPctValue", 5.4, "KM G4"),
    ("pcaOnePct", 47.2, "KM PCA corrected"),
    ("qualDecTwentyFourCdas", 1272207, "QUAL 2024-12"), ("qualDecTwentyFourRef", 2148025, "QUAL 2024-12 ref"),
]
cmp_ = pd.DataFrame([{"key": k, "draft": v, "new": num(k), "new_in_numeros": N[k][0], "draft_source": o} for k, v, o in DRAFT])
cmp_["rel_diff_pct"] = 100 * (cmp_.new - cmp_.draft) / cmp_.draft.abs()
cmp_["matches"] = cmp_.rel_diff_pct.abs() <= 0.5
save_tab(cmp_, "tab_draft_comparison", fmts={"rel_diff_pct": 2}, caption="Draft numbers (27 Oct 2026) vs recomputed")
print(f"match (|diff| <= 0.5%): {cmp_.matches.sum()} of {len(cmp_)}")
cmp_[~cmp_.matches]

**Explaining the differences.** The remaining differences have four sources; none is a change in the data (the database is pinned in section 0).

1. **No 2026 data here; the lab used some.** In the lab, the CDA × year panel behind the Kruskal–Wallis test of the CDA recovery rate ran into 2026 and the payment statistics used GARE payments up to March 2026; that is why the earlier $H$ came out identical to the March 2026 paper. Here the panel stops in 2025 and payments on 31 Dec 2025, so `kwRateH` and the effect sizes based on the rate (`rBisRate`) and on payment timing move; the draft's label "December 2025 snapshot" for those rows was wrong. The K-means payment inputs (recovery, time to payment, number of payments, programme share) are also cut at 31 Dec 2025, which moves the silhouettes and the cluster shares at the margin (`kmSilEuclid`, `kmSilGower`, `kmGTwoPctCda`); the K-means here is re-implemented (100 initialisations per $k$, scikit-learn), which can also change labels at the margin.
2. **Different snapshot for the largest debts in the lab.** The lab's top-set table (B.4.3) and its K-means base were built on the March 2026 snapshot (noted in the lab itself); here the base is the December 2025 snapshot, so the top 1% has 23,222 CDAs instead of 23,253 and the minimum value of the top 1% (`kmMinValueMn`) moves.
3. **Legal-entity definition.** On 27 Oct 2026 the rule became that `CPF_DEVEDOR` and `NOME_DEVEDOR` are never read. A legal entity is therefore a debtor with a 14-character normalised CNPJ, instead of "CNPJ filled and CPF empty" in the draft. This raises the legal-entity share of CDAs (`pctCdaPj`) and can move the legal-entity profile (`medianValueAssessed`, `medianAgeAssessed`, `rBisAge`) slightly. Which part of these small moves comes from this change cannot be checked here without reading the CPF column, which the rule forbids.
4. **A mislabelled draft number.** The draft’s “1.8% of assessed CDAs in the rest of the portfolio” compared the top 0.1% (93.9%) with the remainder **outside the top 1%**. Following the author’s decision of 27 Oct 2026, the article’s “rest” is always what lies outside the top 1% (`restOnePctCdaAssessed`, compared with the draft’s 1.792); no remainder outside the top 0.1% is computed.

Rows still flagged after these explanations are differences of rounding in the draft's own precision (for example, rates reported with two or three decimals).

### 11.1 Presentation tables for the manuscript

The tables above keep the raw column names of the analysis. The manuscript (L6) needs short tables with readable headers; the cell below builds them **only from the CSV files this notebook has just written** (no new computation of the data), with the prefix `tab_ms_`. The full tables stay in `tables/` for the appendix and for replication.

In [ ]:
def rd(name):
    return pd.read_csv(TAB / f"{name}.csv")

FILING_EN = {"Ajuizado": "filed in court", "Liberado para ajuizamento": "cleared for filing", "Inscrito": "enrolled", "Suspenso": "suspended"}

# 1. profile of the two origins
pf_ = rd("tab_profile_type").set_index("type")
bs_ = rd("tab_bases_dec2025").set_index("type")
cols_ = {"Assessed": "Assessed (ICMS Autuação)", "Declared": "Declared (ICMS Declarado)"}
rows_ = [("CDAs, all debtors (Dec 2025)", bs_, "cdas", 0), ("Stock, BRL bn (Dec 2025)", bs_, "stock_bn", 1),
         ("Share of VAT (ICMS) stock, %", bs_, "pct_stock_vat", 1),
         ("Mean CDA value, BRL", pf_, "mean_value", 0), ("Median CDA value, BRL", pf_, "median_value", 0),
         ("Median age, years", pf_, "median_age", 2), ("CDAs filed in court, %", pf_, "pct_cdas_filed", 1),
         ("Stock filed in court, %", pf_, "pct_stock_filed", 1), ("CDAs with any payment, % (2016-2025)", pf_, "pct_any_payment", 1),
         ("Median years to first payment", pf_, "median_years_to_first_payment", 2)]
ms_profile = pd.DataFrame([{"measure": lab, **{k: fmt(src_.loc[v, col], nd) for k, v in cols_.items()}} for lab, src_, col, nd in rows_])
save_tab(ms_profile, "tab_ms_profile", caption="Profile of the two origins; value, age and filing in the December 2025 snapshot (legal entities); payment in the 2016-2025 panel")

# 2. concentration by debtor (CNPJ root)
cc_ = rd("tab_concentration")
cc_ = cc_[cc_.level == "Debtor (CNPJ root)"].copy()
ms_conc = pd.DataFrame({"group": cc_.group, "debtors": cc_.units, "stock (BRL bn)": cc_.stock_bn, "Gini": cc_.gini,
                        "top 0.1% (%)": cc_["top_0.1%"], "top 1% (%)": cc_["top_1%"], "top 10% (%)": cc_["top_10%"], "k50": cc_.k50})
save_tab(ms_conc, "tab_ms_concentration", caption="Concentration of the stock by debtor (CNPJ root), December 2025, all debtors with a CNPJ",
         fmts={"debtors": 0, "stock (BRL bn)": 1, "Gini": 3, "top 0.1% (%)": 1, "top 1% (%)": 1, "top 10% (%)": 1, "k50": 0})

# 3. composition of the top (rest = outside the top 1%)
tc_ = rd("tab_top_composition")
tc_ = tc_[tc_.origin == "assessed"]
ms_top = pd.DataFrame({"set of CDAs": tc_.cut, "assessed CDAs": tc_.cdas, "assessed share of CDAs (%)": tc_.pct_cdas,
                       "assessed share of value (%)": tc_.pct_value})
save_tab(ms_top, "tab_ms_top_composition", caption="Assessed share among the largest legal-entity CDAs, December 2025; rest = outside the top 1%",
         fmts={"assessed CDAs": 0, "assessed share of CDAs (%)": 1, "assessed share of value (%)": 1})

# 4. K-means profiles (k = 4)
kp_ = rd("tab_kmeans_k4_profiles")
ms_km = pd.DataFrame({"cluster": kp_.cluster, "CDAs (%)": kp_.pct_cdas, "value (%)": kp_.pct_value,
                      "modal origin": kp_.origin_mode + " (" + kp_.pct_origin_mode.map(lambda v: fmt(v, 1)) + "%)",
                      "modal situation": kp_.situation_mode.map(lambda x: FILING_EN.get(x, x)),
                      "modal filing status": kp_.filing_mode.map(lambda x: FILING_EN.get(x, x)),
                      "median value (BRL mn)": kp_.median_value_mn, "median age (years)": kp_.median_age, "any payment (%)": kp_.pct_paid})
save_tab(ms_km, "tab_ms_kmeans", caption="K-means (k = 4) on the top 1% of legal-entity CDAs, December 2025; modal labels translated from the raw database values",
         fmts={"CDAs (%)": 1, "value (%)": 1, "median value (BRL mn)": 2, "median age (years)": 1, "any payment (%)": 1})

# 5. registration status by origin (descriptors only)
sr_ = rd("tab_status_recovery")
kt_ = rd("tab_km_status_origin")[["origin", "status", "pct_paid_12m", "pct_paid_60m"]]
m_ = sr_.merge(kt_, on=["origin", "status"], how="left")
ms_st = pd.DataFrame({"origin": m_.origin, "status (Mar 2026)": m_.status, "CNPJs": m_.cnpjs_dec2025,
                      "share of origin stock (%)": m_.pct_origin_stock_dec2025, "any payment (%)": m_.pct_cdas_any_payment,
                      "pooled recovery (%)": m_.pooled_recovery_pct, "KM 12 m (%)": m_.pct_paid_12m, "KM 60 m (%)": m_.pct_paid_60m})
save_tab(ms_st, "tab_ms_status", caption="Registration status of the debtor CNPJ (RFB, March 2026) by origin, descriptors only; KM = Kaplan-Meier share with a first payment, enrolment cohort 2016-2025",
         fmts={"CNPJs": 0, "share of origin stock (%)": 1, "any payment (%)": 1, "pooled recovery (%)": 2, "KM 12 m (%)": 1, "KM 60 m (%)": 1})

# 6. CNAE divisions: the ten largest by December 2025 stock, English names
cd_ = rd("tab_cnae_division_origin")
cd_ = cd_[cd_.division.str.startswith("div ")].head(10)
ms_cnae = pd.DataFrame({"division": cd_.division.str.split().str[1], "activity (CNAE 2.0, English)": cd_.division_name,
                        "share of VAT stock (%)": cd_.pct_vat_stock_dec2025, "assessed weight (%)": cd_.assessed_weight_pct,
                        "recovery, assessed (%)": cd_.recovery_assessed_pct, "recovery, declared (%)": cd_.recovery_declared_pct})
save_tab(ms_cnae, "tab_ms_cnae_top10", caption="Ten largest CNAE divisions by December 2025 VAT (ICMS) stock: assessed weight and pooled 2016-2025 recovery by origin; descriptors only",
         fmts={"share of VAT stock (%)": 1, "assessed weight (%)": 1, "recovery, assessed (%)": 2, "recovery, declared (%)": 2})
reg("cnaeTenPctStock", cd_.pct_vat_stock_dec2025.sum(), 1, "tab_ms_cnae_top10")
print(sorted(p.name for p in TAB.glob("tab_ms_*.tex")))

## 12. `numeros.tex`

Each key registered in `N` becomes a `\newcommand` in `projects/macro/manuscript/icms_declarado_autuado/numeros.tex`, with its source as a comment. The article's LaTeX uses only these macros; no number is typed into the text.

In [ ]:
tex_lines = ["% generated by projects/macro/notebooks/articles/icms_declarado_autuado.ipynb; do not edit by hand",
             f"% extracao={EXTRACAO}; snapshot {SNAP[0]}-{SNAP[1]:02d}; panel {YEAR_MIN}-{YEAR_MAX}; payments < {PAY_END}; seed {SEED}",
             f"% database: cemepi.duckdb ({DB_SEEN['bytes']} bytes, mtime {DB_SEEN['mtime_epoch']}); aggregates only"]
for k in sorted(N):
    txt, source = N[k]
    tex_lines.append(f"\\newcommand{{\\{k}}}{{{txt}}}" + (f"  % {source}" if source else ""))
(OUT / "numeros.tex").write_text("\n".join(tex_lines) + "\n", encoding="utf-8")
print(f"{len(N)} macros in {OUT.relative_to(REPO)}/numeros.tex")

## 13. Open questions **[ÉTORE DECIDE]**

1. Definition of *declared* with PGE (self-assessed and unpaid, or another modality).
2. Research questions RQ-A to RQ-C (section 8.1) on the 2020 fall and the 2019 and 2024 spikes in assessed debt: to investigate and re-run with `DEBITO` restored; no cause is asserted here.
3. Joint reading of the concentration measures: assessed debt holds most of the VAT (ICMS) stock, but within each origin the top 1% of debtors may weigh more in declared debt.
4. Weight of the without-origin sensitivity (section 9.3) in any claim that origin is the axis separating the largest debts.
5. Programme windows (PEP 2019, Acordo Paulista 2024) used in the K-means.
6. Method next steps: tests at the debtor level or with resampling by debtor.
7. Firm characteristics (section 10): decided on 27 Oct 2026 — registration status and CNAE enter only as descriptors, with English division names; still open: which divisions, if any, the text names.

## References

- Allingham, M. G., & Sandmo, A. (1972). Income tax evasion: a theoretical analysis. *Journal of Public Economics*, 1(3–4), 323–338. https://doi.org/10.1016/0047-2727(72)90010-2 (in `docs/references/referencias_grounded.md`).
- Arnold, J. M., Battiau, P., Fall, F., & Spies, K. (2025). *The reform of Brazil's consumption tax system*. OECD Economics Department Working Papers, No. 1848. OECD Publishing. https://doi.org/10.1787/2a85e559-en
- Benjamini, Y., & Hochberg, Y. (1995). Controlling the false discovery rate: a practical and powerful approach to multiple testing. *Journal of the Royal Statistical Society: Series B*, 57(1), 289–300. https://doi.org/10.1111/j.2517-6161.1995.tb02031.x
- Cabello, O. G., Gaio, L. E., & Rezende, A. J. (2020). State value-added tax collection efficiency in Brazil. *Advances in Scientific and Applied Accounting*, 22–38. https://doi.org/10.14392/asaa.2019120302
- Gomes, G. S. L., & Balaniuk, R. (2024). Identifying tax evasion in shell companies and fraudulent credits of Brazil’s state value-added tax (ICMS). *Revista de Administração Pública*, 58(4), e2023-0303. https://doi.org/10.1590/0034-761220230303x
- Gower, J. C. (1971). A general coefficient of similarity and some of its properties. *Biometrics*, 27(4), 857–871. https://doi.org/10.2307/2528823
- Keen, M., & Smith, S. (2006). VAT fraud and evasion: what do we know and what can be done? *National Tax Journal*, 59(4), 861–887. https://doi.org/10.17310/ntj.2006.4.07
- Kaplan, E. L., & Meier, P. (1958). Nonparametric estimation from incomplete observations. *Journal of the American Statistical Association*, 53(282), 457–481. https://doi.org/10.1080/01621459.1958.10501452
- Kerby, D. S. (2014). The simple difference formula: an approach to teaching nonparametric correlation. *Comprehensive Psychology*, 3, 11.IT.3.1. https://doi.org/10.2466/11.IT.3.1
- Kleven, H. J., Knudsen, M. B., Kreiner, C. T., Pedersen, S., & Saez, E. (2011). Unwilling or unable to cheat? Evidence from a tax audit experiment in Denmark. *Econometrica*, 79(3), 651–692. https://doi.org/10.3982/ECTA9113
- Kruskal, W. H., & Wallis, W. A. (1952). Use of ranks in one-criterion variance analysis. *Journal of the American Statistical Association*, 47(260), 583–621. https://doi.org/10.1080/01621459.1952.10483441
- Lloyd, S. (1982). Least squares quantization in PCM. *IEEE Transactions on Information Theory*, 28(2), 129–137. https://doi.org/10.1109/TIT.1982.1056489
- Lorenz, M. O. (1905). Methods of measuring the concentration of wealth. *Publications of the American Statistical Association*, 9(70), 209–219. https://doi.org/10.1080/15225437.1905.10503443 (the older JSTOR DOI 10.2307/2276207 redirects to it)
- Mann, H. B., & Whitney, D. R. (1947). On a test of whether one of two random variables is stochastically larger than the other. *The Annals of Mathematical Statistics*, 18(1), 50–60. https://doi.org/10.1214/aoms/1177730491
- OECD (2026). The OECD classification of taxes and interpretative guide. In *Revenue Statistics in Latin America and the Caribbean 2026*. OECD Publishing. https://www.oecd.org/en/publications/revenue-statistics-in-latin-america-and-the-caribbean-2026_2f21f131-en/full-report/the-oecd-classification-of-taxes-and-interpretative-guide_d70fc5cb.html (official source; §61, heading 5111).
- IBGE (2015). *Classificação Nacional de Atividades Econômicas — CNAE, versão 2.0*, 2nd edition. IBGE/CONCLA; structure retrieved from https://servicodados.ibge.gov.br/api/v2/cnae/divisoes on 27 Oct 2026 (division names translated literally into English by the authors).
- Pomeranz, D. (2015). No taxation without information: deterrence and self-enforcement in the value added tax. *American Economic Review*, 105(8), 2539–2569. https://doi.org/10.1257/aer.20130393
- Rousseeuw, P. J. (1987). Silhouettes: a graphical aid to the interpretation and validation of cluster analysis. *Journal of Computational and Applied Mathematics*, 20, 53–65. https://doi.org/10.1016/0377-0427(87)90125-7
- Spearman, C. (1904). The proof and measurement of association between two things. *The American Journal of Psychology*, 15(1), 72–101. https://doi.org/10.2307/1412159

References outside `referencias_grounded.md` were checked by DOI on Crossref on 27 Oct 2026 (Lorenz also on OpenAlex and through the doi.org redirect; the Arnold et al. quote was checked against the PDF).

*Use of AI:* an AI assistant helped assemble this notebook; the analysis, the flagged decisions and the conclusions are the author's.

In [ ]:
con.close()
shutil.rmtree(TMP, ignore_errors=True)
print("outputs:", sorted(str(p.relative_to(OUT)) for p in OUT.rglob("*") if p.is_file()))